reqisits

In [ ]:
!pip install fyers-apiv3 polars pyarrow requests --quiet
print("✅ Step 1 Complete: Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 50.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.8/171.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.0/804.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.2 MB/s eta 

verification

In [10]:
import hashlib
import requests
import urllib.parse
import polars as pl

# -------------------------------------------------------------------
# 1. CREDENTIALS (Configured with Non-Trading App: Code testing)
# -------------------------------------------------------------------
APP_ID = "4BG0I942OL-100"
SECRET_KEY = "LQ14AUD71Q"
REDIRECT_URI = "https://trade.fyers.in/api-login/redirect-uri/index.html"

# Compute SHA-256 Hash for auth code exchange
string_to_hash = f"{APP_ID}:{SECRET_KEY}"
app_id_hash = hashlib.sha256(string_to_hash.encode("utf-8")).hexdigest()

# -------------------------------------------------------------------
# 2. GENERATE AUTH URL
# -------------------------------------------------------------------
auth_url = (
    f"https://api-t1.fyers.in/api/v3/generate-authcode?"
    f"client_id={APP_ID}&redirect_uri={REDIRECT_URI}&response_type=code&state=None"
)

print(f"👉 STEP 1: Open this link in your browser to log in:\n{auth_url}\n")
user_input = input("👉 STEP 2: Paste authorization code (or full redirected URL) here: ").strip()

# Extract code cleanly if full redirected URL was pasted
if "auth_code=" in user_input:
    auth_code = urllib.parse.parse_qs(urllib.parse.urlparse(user_input).query)['auth_code'][0]
elif "code=" in user_input:
    auth_code = urllib.parse.parse_qs(urllib.parse.urlparse(user_input).query)['code'][0]
else:
    auth_code = user_input

# -------------------------------------------------------------------
# 3. EXCHANGE AUTH CODE FOR ACCESS TOKEN
# -------------------------------------------------------------------
token_url = "https://api-t1.fyers.in/api/v3/validate-authcode"
payload = {
    "grant_type": "authorization_code",
    "appIdHash": app_id_hash,
    "code": auth_code
}

response = requests.post(token_url, json=payload).json()

if response.get("s") == "ok":
    access_token = response.get("access_token")
    print(f"\n✅ SUCCESS! ACCESS TOKEN GENERATED!")

    # -------------------------------------------------------------------
    # 4. FETCH HISTORICAL 1-MINUTE CANDLE DATA FOR EXPIRED OPTIONS
    # -------------------------------------------------------------------
    print("\n⚡ Fetching historical 1-minute candles for expired option...")

    headers = {
        "Authorization": f"{APP_ID}:{access_token}"
    }

    # Endpoint specifically for EXPIRED F&O Contracts
    expired_fno_url = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

    params = {
        "symbol": "NSE:NIFTY26AUG24500CE", # Target option ticker
        "resolution": "1",                # 1-minute resolution
        "date_format": "1",               # YYYY-MM-DD
        "range_from": "2026-08-01",
        "range_to": "2026-08-31"
    }

    hist_res = requests.get(expired_fno_url, headers=headers, params=params).json()

    if hist_res.get("s") == "ok":
        candles = hist_res.get("candles", [])
        df = pl.DataFrame(
            candles,
            schema=["timestamp", "open", "high", "low", "close", "volume"],
            orient="row"
        )
        df = df.with_columns(pl.from_epoch("timestamp", time_unit="s"))
        print("\n✅ Successfully retrieved 1-minute candles into Polars DataFrame:")
        print(df.head(10))
    else:
        print(f"\n⚠️ Candle Fetch Error: {hist_res}")

else:
    print(f"\n❌ Token Generation Failed: {response}")

👉 STEP 1: Open this link in your browser to log in:
https://api-t1.fyers.in/api/v3/generate-authcode?client_id=4BG0I942OL-100&redirect_uri=https://trade.fyers.in/api-login/redirect-uri/index.html&response_type=code&state=None

👉 STEP 2: Paste authorization code (or full redirected URL) here: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhcHBfaWQiOiI0QkcwSTk0Mk9MIiwidXVpZCI6IjgxZjI3MzE0ZGE2YTRmNmZiMDFmOWNmMzYxNTNhZjkyIiwiaXBBZGRyIjoiIiwibm9uY2UiOiIiLCJzY29wZSI6IiIsImRpc3BsYXlfbmFtZSI6IkZBSzk1NzYyIiwib21zIjoiSzEiLCJoc21fa2V5IjoiZWUxOTVjMGU4MmYzN2E1YTRjZmFjYzNhYjA5M2U0YWRlZGRlMzEzYzEyMGE0NWRiNGUyYmJiZWIiLCJpc0RkcGlFbmFibGVkIjoiTiIsImlzTXRmRW5hYmxlZCI6Ik4iLCJhdWQiOiJbXCJkOjFcIixcImQ6MlwiLFwieDowXCIsXCJ4OjFcIl0iLCJleHAiOjE3OTEzMjExNzMsImlhdCI6MTc5MTI5MTE3MywiaXNzIjoiYXBpLmxvZ2luLmZ5ZXJzLmluIiwibmJmIjoxNzkxMjkxMTczLCJzdWIiOiJhdXRoX2NvZGUifQ.wHnNCHmpJuPtEnIl6xWz-J8kXJwgVBu3H1eHCGlHDQs

✅ SUCCESS! ACCESS TOKEN GENERATED!

⚡ Fetching historical 1-minute candles for expired option...

✅ Successfully r

In [ ]:
import os
import requests
import polars as pl

# Month Configuration
YEAR = "2026"
MONTH = "08"
START_DATE = f"{YEAR}-{MONTH}-01"
END_DATE = f"{YEAR}-{MONTH}-31"

BASE_DIR = "fyers_backtest_data"
SPOT_DIR = os.path.join(BASE_DIR, "spot", f"year={YEAR}", f"month={MONTH}")
os.makedirs(SPOT_DIR, exist_ok=True)

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}

spot_url = "https://api-t1.fyers.in/data/history"
params = {
    "symbol": "NSE:NIFTY50-INDEX",
    "resolution": "1",
    "date_format": "1",
    "range_from": START_DATE,
    "range_to": END_DATE,
    "cont_flag": "1"
}

print(f"⚡ Downloading Nifty Spot 1-min candles for {YEAR}-{MONTH}...")
res = requests.get(spot_url, headers=HEADERS, params=params).json()

if res.get("s") == "ok" and res.get("candles"):
    df_spot = pl.DataFrame(
        res["candles"],
        schema=["epoch", "open", "high", "low", "close", "volume"],
        orient="row"
    )

    df_spot = df_spot.with_columns([
        pl.from_epoch("epoch", time_unit="s")
          .dt.replace_time_zone("UTC")
          .dt.convert_time_zone("Asia/Kolkata")
          .alias("timestamp"),
        (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
        (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
        (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
        (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32"),
        pl.lit("NIFTY50").cast(pl.Categorical).alias("symbol")
    ]).select(["timestamp", "symbol", "open_i32", "high_i32", "low_i32", "close_i32", "volume"])

    spot_file = os.path.join(SPOT_DIR, "NIFTY50.parquet")
    df_spot.write_parquet(spot_file, compression="zstd", compression_level=3)

    min_p = df_spot["close_i32"].min() / 100.0
    max_p = df_spot["close_i32"].max() / 100.0
    print(f"✅ Saved Nifty Spot data ({len(df_spot)} candles) to: {spot_file}")
    print(f"📈 Spot Range: Min = {min_p:.2f}, Max = {max_p:.2f}")
else:
    print(f"❌ Error fetching spot data: {res}")

⚡ Downloading Nifty Spot 1-min candles for 2026-08...
✅ Saved Nifty Spot data (7875 candles) to: fyers_backtest_data/spot/year=2026/month=08/NIFTY50.parquet
📈 Spot Range: Min = 23994.90, Max = 24774.30


In [ ]:
import re

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"

# 1. Fetch official exchange expiry dates from FYERS
params_exp = {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": START_DATE,
    "range_to": END_DATE,
    "date_format": "1"
}

res_exp = requests.get(EXPIRY_DATES_URL, headers=HEADERS, params=params_exp).json()

if res_exp.get("s") == "ok" and "data" in res_exp and "expiry_dates" in res_exp["data"]:
    api_expiries = sorted(res_exp["data"]["expiry_dates"].get("options", []))
else:
    raise ValueError(f"Failed to parse expiries from FYERS API: {res_exp}")

monthly_expiry_date = api_expiries[-1]  # Final expiry of month = Monthly contract

print(f"📅 Official FYERS Option Expiry Dates for {YEAR}-{MONTH}:")

df_spot_lazy = pl.scan_parquet(spot_file)
expiry_contract_map = {}

for exp_date in api_expiries:
    exp_type = "MONTHLY" if exp_date == monthly_expiry_date else "WEEKLY"

    # Derive ATM Strike from Spot Closing Price
    spot_candle = (
        df_spot_lazy
        .filter(pl.col("timestamp").dt.date() == pl.lit(exp_date).str.to_date())
        .select("close_i32")
        .last()
        .collect()
    )

    spot_price = spot_candle["close_i32"][0] / 100.0 if not spot_candle.is_empty() else 24500.0
    atm_strike = int(round(spot_price / 50.0) * 50)

    # Build 8 ITM + 1 ATM + 10 OTM target strike sets
    ce_itm_strikes = [atm_strike - (i * 50) for i in range(8, 0, -1)]  # 8 ITM CE
    ce_otm_strikes = [atm_strike + (i * 50) for i in range(1, 11)]     # 10 OTM CE
    pe_itm_strikes = [atm_strike + (i * 50) for i in range(8, 0, -1)]  # 8 ITM PE
    pe_otm_strikes = [atm_strike - (i * 50) for i in range(1, 11)]     # 10 OTM PE

    target_strikes = set(ce_itm_strikes + [atm_strike] + ce_otm_strikes + pe_itm_strikes + pe_otm_strikes)

    # 2. Query option contracts active on this expiry date
    sym_params = {
        "symbol": "NSE:NIFTY50-INDEX",
        "expiry_date": exp_date
    }
    sym_res = requests.get(UNDERLYING_SYMBOLS_URL, headers=HEADERS, params=sym_params).json()

    exp_contracts = []

    # Extract option contract list from data -> contracts -> options
    if sym_res.get("s") == "ok" and "data" in sym_res:
        data_block = sym_res["data"]
        raw_options = data_block.get("contracts", {}).get("options", [])

        for symbol_name in raw_options:
            match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
            if match and int(match.group(1)) in target_strikes:
                exp_contracts.append(symbol_name)

    expiry_contract_map[exp_date] = {
        "type": exp_type,
        "atm_strike": atm_strike,
        "contracts": sorted(list(set(exp_contracts)))
    }

    print(f"  • {exp_date} [{exp_type}] ➔ Spot: {spot_price:.2f} | ATM: {atm_strike} | Matched Contracts: {len(exp_contracts)}")

total_contracts = sum(len(v["contracts"]) for v in expiry_contract_map.values())
print(f"\n📦 Total Option Contracts Matched across Month: {total_contracts}")

📅 Official FYERS Option Expiry Dates for 2026-08:
  • 2026-08-04 [WEEKLY] ➔ Spot: 24614.90 | ATM: 24600 | Matched Contracts: 42
  • 2026-08-11 [WEEKLY] ➔ Spot: 24471.70 | ATM: 24450 | Matched Contracts: 42
  • 2026-08-18 [WEEKLY] ➔ Spot: 24154.90 | ATM: 24150 | Matched Contracts: 42
  • 2026-08-25 [MONTHLY] ➔ Spot: 24334.55 | ATM: 24350 | Matched Contracts: 42

📦 Total Option Contracts Matched across Month: 168


In [ ]:
import os
import time
import requests
import polars as pl

OPTIONS_HIVE_DIR = os.path.join(BASE_DIR, "options", f"year={YEAR}", f"month={MONTH}")
os.makedirs(OPTIONS_HIVE_DIR, exist_ok=True)

EXPIRED_HIST_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"
success_count = 0

for exp_date, info in expiry_contract_map.items():
    exp_type = info["type"]
    contracts = info["contracts"]
    print(f"\n🚀 Downloading {exp_type} Expiry ({exp_date}) - {len(contracts)} contracts...")

    for i, symbol in enumerate(contracts, 1):
        params = {
            "symbol": symbol,
            "resolution": "1",
            "date_format": "1",
            "range_from": START_DATE,
            "range_to": END_DATE
        }

        try:
            res = requests.get(EXPIRED_HIST_URL, headers=HEADERS, params=params).json()

            if res.get("s") == "ok" and res.get("candles"):
                candles = res["candles"]

                df_raw = pl.DataFrame(
                    candles,
                    schema=["epoch", "open", "high", "low", "close", "volume"],
                    orient="row"
                )

                df_scaled = df_raw.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
                    (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
                    (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32"),
                    pl.lit(symbol.replace("NSE:", "")).cast(pl.Categorical).alias("symbol"),
                    pl.lit(exp_type).cast(pl.Categorical).alias("expiry_type"),
                    pl.lit(exp_date).cast(pl.Categorical).alias("expiry_date")
                ]).select([
                    "timestamp", "symbol", "expiry_type", "expiry_date",
                    "open_i32", "high_i32", "low_i32", "close_i32", "volume"
                ])

                clean_symbol = symbol.replace("NSE:", "")
                output_path = os.path.join(OPTIONS_HIVE_DIR, f"{clean_symbol}.parquet")
                df_scaled.write_parquet(output_path, compression="zstd", compression_level=3)

                success_count += 1
                print(f"[{i}/{len(contracts)}] ✅ Saved: {clean_symbol} ({len(df_scaled)} rows)")
            else:
                print(f"[{i}/{len(contracts)}] ⚠️ No data for {symbol}")

        except Exception as e:
            print(f"[{i}/{len(contracts)}] ❌ Exception: {symbol} - {e}")

        time.sleep(0.12)

print(f"\n🎉 Download Complete! Saved {success_count} option files into '{OPTIONS_HIVE_DIR}'")


🚀 Downloading WEEKLY Expiry (2026-08-04) - 42 contracts...
[1/42] ✅ Saved: NIFTY2680424100CE (770 rows)
[2/42] ✅ Saved: NIFTY2680424100PE (770 rows)
[3/42] ✅ Saved: NIFTY2680424150CE (770 rows)
[4/42] ✅ Saved: NIFTY2680424150PE (770 rows)
[5/42] ✅ Saved: NIFTY2680424200CE (770 rows)
[6/42] ✅ Saved: NIFTY2680424200PE (770 rows)
[7/42] ✅ Saved: NIFTY2680424250CE (770 rows)
[8/42] ✅ Saved: NIFTY2680424250PE (770 rows)
[9/42] ✅ Saved: NIFTY2680424300CE (770 rows)
[10/42] ✅ Saved: NIFTY2680424300PE (770 rows)
[11/42] ✅ Saved: NIFTY2680424350CE (770 rows)
[12/42] ✅ Saved: NIFTY2680424350PE (770 rows)
[13/42] ✅ Saved: NIFTY2680424400CE (770 rows)
[14/42] ✅ Saved: NIFTY2680424400PE (770 rows)
[15/42] ✅ Saved: NIFTY2680424450CE (770 rows)
[16/42] ✅ Saved: NIFTY2680424450PE (770 rows)
[17/42] ✅ Saved: NIFTY2680424500CE (770 rows)
[18/42] ✅ Saved: NIFTY2680424500PE (770 rows)
[19/42] ✅ Saved: NIFTY2680424550CE (770 rows)
[20/42] ✅ Saved: NIFTY2680424550PE (770 rows)
[21/42] ✅ Saved: NIFTY2680424

### Comprehensive Option Chain for 'week1' (Latest Prices)

This option chain aggregates the latest available prices (LTP) for all unique call and put strikes found across the entire 'week1' dataset, providing a complete view of the week's options.

In [ ]:
import polars as pl

# 1. Define target timestamp with Asia/Kolkata timezone
target_time = (
    pl.Series(["2026-08-03 09:50:00"])
    .str.to_datetime()
    .dt.replace_time_zone("Asia/Kolkata")[0]
)

# 2. Get Spot Price at 09:50 and calculate ATM strike
spot_0950_i32 = df_spot_check.filter(pl.col("timestamp") == target_time)["close_i32"][0]
spot_price = spot_0950_i32 / 100.0
atm_strike = int(round(spot_price / 50.0) * 50)

print(f"📍 NIFTY Spot at 09:50: {spot_price:.2f} | ATM Strike: {atm_strike}\n")

# 3. Filter week1 options for 09:50 snapshot
snapshot_0950 = (
    df_week1_combined
    .filter(pl.col("timestamp") == target_time)
    .with_columns([
        pl.col("symbol").cast(pl.String).str.extract(r"(\d{5})(CE|PE)$", 1).cast(pl.Int32).alias("strike"),
        pl.col("symbol").cast(pl.String).str.extract(r"(\d{5})(CE|PE)$", 2).alias("option_type"),
        (pl.col("close_i32") / 100.0).alias("ltp")
    ])
)

# 4. Separate Calls (CE) and Puts (PE)
calls_0950 = snapshot_0950.filter(pl.col("option_type") == "CE").select(["strike", pl.col("ltp").alias("Call LTP")])
puts_0950 = snapshot_0950.filter(pl.col("option_type") == "PE").select(["strike", pl.col("ltp").alias("Put LTP")])

# 5. Join Call & Put LTPs and highlight ATM
option_chain_0950 = (
    calls_0950
    .join(puts_0950, on="strike", how="full")
    .with_columns([
        pl.when(pl.col("strike") == atm_strike)
          .then(pl.lit("🎯 ATM"))
          .otherwise(pl.lit(""))
          .alias("ATM")
    ])
    .select([
        pl.col("Call LTP"),
        pl.col("strike").alias("Strike"),
        pl.col("Put LTP"),
        pl.col("ATM")
    ])
    .sort("Strike")
)

print("📊 OPTION CHAIN AT 09:50:00 IST (AUG 03, 2026)\n")
with pl.Config(tbl_rows=-1):
    print(option_chain_0950)

📍 NIFTY Spot at 09:50: 24550.25 | ATM Strike: 24550

📊 OPTION CHAIN AT 09:50:00 IST (AUG 03, 2026)

shape: (21, 4)
┌──────────┬────────┬─────────┬────────┐
│ Call LTP ┆ Strike ┆ Put LTP ┆ ATM    │
│ ---      ┆ ---    ┆ ---     ┆ ---    │
│ f64      ┆ i32    ┆ f64     ┆ str    │
╞══════════╪════════╪═════════╪════════╡
│ 464.0    ┆ 24100  ┆ 4.55    ┆        │
│ 414.65   ┆ 24150  ┆ 5.15    ┆        │
│ 365.95   ┆ 24200  ┆ 6.1     ┆        │
│ 317.25   ┆ 24250  ┆ 7.55    ┆        │
│ 269.95   ┆ 24300  ┆ 9.6     ┆        │
│ 223.1    ┆ 24350  ┆ 12.85   ┆        │
│ 178.05   ┆ 24400  ┆ 18.1    ┆        │
│ 137.1    ┆ 24450  ┆ 26.45   ┆        │
│ 99.9     ┆ 24500  ┆ 39.7    ┆        │
│ 68.35    ┆ 24550  ┆ 58.2    ┆ 🎯 ATM │
│ 44.25    ┆ 24600  ┆ 84.4    ┆        │
│ 27.05    ┆ 24650  ┆ 117.35  ┆        │
│ 16.1     ┆ 24700  ┆ 156.35  ┆        │
│ 9.35     ┆ 24750  ┆ 199.8   ┆        │
│ 5.85     ┆ 24800  ┆ 246.0   ┆        │
│ 3.8      ┆ 24850  ┆ 294.0   ┆        │
│ 2.75     ┆ 24900  ┆ 343

In [ ]:
import os

YEAR = "2026"
MONTH = "08"
BASE_DIR = "fyers_backtest_data"

def get_dir_size(path):
    total_bytes = 0
    count = 0
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith('.parquet'):
                fp = os.path.join(root, f)
                total_bytes += os.path.getsize(fp)
                count += 1
    return total_bytes, count

# 1. Paths
options_dir = os.path.join(BASE_DIR, "options", f"year={YEAR}", f"month={MONTH}")
spot_dir = os.path.join(BASE_DIR, "spot", f"year={YEAR}", f"month={MONTH}")

# 2. Get sizes
opt_bytes, opt_count = get_dir_size(options_dir)
spot_bytes, spot_count = get_dir_size(spot_dir)

# 3. Convert to MB
opt_mb = opt_bytes / (1024 * 1024)
spot_mb = spot_bytes / (1024 * 1024)
total_mb = opt_mb + spot_mb

print(f"📊 STORAGE BREAKDOWN FOR {YEAR}-{MONTH}:")
print(f" ├─ 📈 Spot 1-Min Data:    {spot_mb:.2f} MB ({spot_count} file)")
print(f" ├─ 🎯 Options 1-Min Data: {opt_mb:.2f} MB ({opt_count} files)")
print(f" └─ 💾 Total Combined:     {total_mb:.2f} MB")

📊 STORAGE BREAKDOWN FOR 2026-08:
 ├─ 📈 Spot 1-Min Data:    0.16 MB (1 file)
 ├─ 🎯 Options 1-Min Data: 8.80 MB (168 files)
 └─ 💾 Total Combined:     8.96 MB


In [ ]:
BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"

In [ ]:
import os
import time
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configurable Batch
BATCH_MONTHS = [(2026, "05"), (2026, "06"), (2026, "07")]
BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"

print(f"🚀 Initializing Batch Download for {len(BATCH_MONTHS)} months...\n")

for year, month in BATCH_MONTHS:
    print(f"==================================================")
    print(f"📅 PROCESSING: YEAR {year} | MONTH {month}")
    print(f"==================================================")

    options_base = os.path.join(BASE_DIR, "options", f"year={year}", f"month={month}")
    spot_base = os.path.join(BASE_DIR, "spot", f"year={year}", f"month={month}")

    os.makedirs(options_base, exist_ok=True)
    os.makedirs(spot_base, exist_ok=True)

    # Check Spot File
    spot_file = os.path.join(spot_base, "NIFTY50_SPOT.parquet")
    if not os.path.exists(spot_file):
        print("📈 Requesting Spot 1-Min Data from FYERS...")
        # Ensure your actual FYERS API call method (e.g., fyers.history(...)) is un-commented here!
        time.sleep(0.3)
    else:
        print("⚡ Spot Data file already present in Google Drive. Skipping.")

    print(f"✅ Month {year}-{month} check completed!\n")

print("🎉 Batch run finished successfully!")

Mounted at /content/drive
🚀 Initializing Batch Download for 3 months...

📅 PROCESSING: YEAR 2026 | MONTH 05
📈 Requesting Spot 1-Min Data from FYERS...
✅ Month 2026-05 check completed!

📅 PROCESSING: YEAR 2026 | MONTH 06
📈 Requesting Spot 1-Min Data from FYERS...
✅ Month 2026-06 check completed!

📅 PROCESSING: YEAR 2026 | MONTH 07
📈 Requesting Spot 1-Min Data from FYERS...
✅ Month 2026-07 check completed!

🎉 Batch run finished successfully!


In [ ]:
import os
import time
import requests
import polars as pl

# ----------------------------------------------------
# A. CONFIGURATION FOR JULY 2026
# ----------------------------------------------------
YEAR = "2026"
MONTH = "07"
START_DATE = f"{YEAR}-{MONTH}-01"
END_DATE = f"{YEAR}-{MONTH}-31"

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_HIVE_DIR = os.path.join(BASE_DIR, "options", f"year={YEAR}", f"month={MONTH}")
os.makedirs(OPTIONS_HIVE_DIR, exist_ok=True)

EXPIRED_HIST_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

# ----------------------------------------------------
# B. JULY 2026 EXPIRY MAP (TUESDAY MONTHLY ON JULY 28)
# ----------------------------------------------------
expiries_jul = {
    "2026-07-07": {"type": "week1", "code": "26707"},
    "2026-07-14": {"type": "week2", "code": "26714"},
    "2026-07-21": {"type": "week3", "code": "26721"},
    "2026-07-28": {"type": "monthly", "code": "26JUL"}  # Monthly contract on last Tuesday
}

strikes = range(23500, 26000, 50)  # Nifty Strike range

expiry_contract_map = {}
for exp_date, info in expiries_jul.items():
    contracts = []
    code = info["code"]
    for strike in strikes:
        contracts.append(f"NSE:NIFTY{code}{strike}CE")
        contracts.append(f"NSE:NIFTY{code}{strike}PE")
    expiry_contract_map[exp_date] = {
        "type": info["type"],
        "contracts": contracts
    }

# ----------------------------------------------------
# C. DOWNLOADER LOOP
# ----------------------------------------------------
success_count = 0

for exp_date, info in expiry_contract_map.items():
    exp_type = info["type"]
    contracts = info["contracts"]
    range_to_date = exp_date

    print(f"\n🚀 Downloading {exp_type} Expiry ({exp_date}) - {len(contracts)} contracts...")

    for i, symbol in enumerate(contracts, 1):
        clean_symbol = symbol.replace("NSE:", "")
        output_path = os.path.join(OPTIONS_HIVE_DIR, f"{clean_symbol}.parquet")

        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print(f"[{i}/{len(contracts)}] ⚡ Skipping existing: {clean_symbol}")
            success_count += 1
            continue

        params = {
            "symbol": symbol,
            "resolution": "1",
            "date_format": "1",
            "range_from": START_DATE,
            "range_to": range_to_date
        }

        try:
            res = requests.get(EXPIRED_HIST_URL, headers=HEADERS, params=params).json()

            if res.get("s") == "ok" and res.get("candles"):
                candles = res["candles"]

                df_raw = pl.DataFrame(
                    candles,
                    schema=["epoch", "open", "high", "low", "close", "volume"],
                    orient="row"
                )

                df_scaled = df_raw.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
                    (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
                    (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32"),
                    pl.lit(clean_symbol).cast(pl.Categorical).alias("symbol"),
                    pl.lit(exp_type).cast(pl.Categorical).alias("expiry_type"),
                    pl.lit(exp_date).cast(pl.Categorical).alias("expiry_date")
                ]).select([
                    "timestamp", "symbol", "expiry_type", "expiry_date",
                    "open_i32", "high_i32", "low_i32", "close_i32", "volume"
                ])

                df_scaled.write_parquet(output_path, compression="zstd", compression_level=3)
                success_count += 1
                print(f"[{i}/{len(contracts)}] ✅ Saved: {clean_symbol} ({len(df_scaled)} rows)")
            else:
                print(f"[{i}/{len(contracts)}] ⚠️ No data for {symbol}")

        except Exception as e:
            print(f"[{i}/{len(contracts)}] ❌ Exception: {symbol} - {e}")

        time.sleep(0.12)

print(f"\n🎉 July 2026 Download Complete! Saved {success_count} option files into '{OPTIONS_HIVE_DIR}'")


🚀 Downloading week1 Expiry (2026-07-07) - 100 contracts...
[1/100] ✅ Saved: NIFTY2670723500CE (1875 rows)
[2/100] ✅ Saved: NIFTY2670723500PE (1875 rows)
[3/100] ✅ Saved: NIFTY2670723550CE (1863 rows)
[4/100] ✅ Saved: NIFTY2670723550PE (1875 rows)
[5/100] ✅ Saved: NIFTY2670723600CE (1875 rows)
[6/100] ✅ Saved: NIFTY2670723600PE (1875 rows)
[7/100] ✅ Saved: NIFTY2670723650CE (1875 rows)
[8/100] ✅ Saved: NIFTY2670723650PE (1875 rows)
[9/100] ✅ Saved: NIFTY2670723700CE (1875 rows)
[10/100] ✅ Saved: NIFTY2670723700PE (1875 rows)
[11/100] ✅ Saved: NIFTY2670723750CE (1873 rows)
[12/100] ✅ Saved: NIFTY2670723750PE (1875 rows)
[13/100] ✅ Saved: NIFTY2670723800CE (1875 rows)
[14/100] ✅ Saved: NIFTY2670723800PE (1875 rows)
[15/100] ✅ Saved: NIFTY2670723850CE (1875 rows)
[16/100] ✅ Saved: NIFTY2670723850PE (1875 rows)
[17/100] ✅ Saved: NIFTY2670723900CE (1875 rows)
[18/100] ✅ Saved: NIFTY2670723900PE (1875 rows)
[19/100] ✅ Saved: NIFTY2670723950CE (1875 rows)
[20/100] ✅ Saved: NIFTY2670723950PE (

In [ ]:
import os
import shutil
from google.colab import drive

# 1. Force remount to ensure Google Drive paths are updated
drive.mount('/content/drive', force_remount=True)

# 2. Base directory setup
BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
MONTH_DIR = os.path.join(BASE_DIR, "options", "year=2026", "month=07")

# Ensure subfolders exist in month=07
folders = ["week1", "week2", "week3", "monthly"]
for f in folders:
    os.makedirs(os.path.join(MONTH_DIR, f), exist_ok=True)

# Contract symbol codes to target folder mapping
code_to_folder = {
    "26707": "week1",
    "26714": "week2",
    "26721": "week3",
    "26JUL": "monthly"
}

# 3. Scan and move files from month=07 root into subfolders
files_moved = 0
items = os.listdir(MONTH_DIR)

for item in items:
    src_path = os.path.join(MONTH_DIR, item)

    # Process only Parquet files sitting directly in month=07
    if os.path.isfile(src_path) and item.endswith(".parquet"):
        for code, subfolder in code_to_folder.items():
            if code in item:
                dest_dir = os.path.join(MONTH_DIR, subfolder)
                dest_path = os.path.join(dest_dir, item)

                shutil.move(src_path, dest_path)
                files_moved += 1
                print(f"📦 Moved {item} ➔ /{subfolder}/")
                break

print(f"\n✅ Migration Complete! Moved {files_moved} files into their proper subfolders.")

Mounted at /content/drive
📦 Moved NIFTY2670723500CE.parquet ➔ /week1/
📦 Moved NIFTY2670723500PE.parquet ➔ /week1/
📦 Moved NIFTY2670723550CE.parquet ➔ /week1/
📦 Moved NIFTY2670723550PE.parquet ➔ /week1/
📦 Moved NIFTY2670723600CE.parquet ➔ /week1/
📦 Moved NIFTY2670723600PE.parquet ➔ /week1/
📦 Moved NIFTY2670723650CE.parquet ➔ /week1/
📦 Moved NIFTY2670723650PE.parquet ➔ /week1/
📦 Moved NIFTY2670723700CE.parquet ➔ /week1/
📦 Moved NIFTY2670723700PE.parquet ➔ /week1/
📦 Moved NIFTY2670723750CE.parquet ➔ /week1/
📦 Moved NIFTY2670723750PE.parquet ➔ /week1/
📦 Moved NIFTY2670723800CE.parquet ➔ /week1/
📦 Moved NIFTY2670723800PE.parquet ➔ /week1/
📦 Moved NIFTY2670723850CE.parquet ➔ /week1/
📦 Moved NIFTY2670723850PE.parquet ➔ /week1/
📦 Moved NIFTY2670723900CE.parquet ➔ /week1/
📦 Moved NIFTY2670723900PE.parquet ➔ /week1/
📦 Moved NIFTY2670723950CE.parquet ➔ /week1/
📦 Moved NIFTY2670723950PE.parquet ➔ /week1/
📦 Moved NIFTY2670724000CE.parquet ➔ /week1/
📦 Moved NIFTY2670724000PE.parquet ➔ /week1/
📦 Move

In [ ]:
import os
import glob
import shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Existing Base Paths
DRIVE_BASE = "/content/drive/MyDrive/fyers_backtest_data"
DRIVE_OPTIONS_AUG = os.path.join(DRIVE_BASE, "options", "year=2026", "month=08")
DRIVE_SPOT_DIR = os.path.join(DRIVE_BASE, "spot", "year=2026")

# Expiry mapping for August 2026 (Tuesday Expiries: Aug 4, 11, 18, 25)
code_to_folder = {
    "26804": "week1",
    "26811": "week2",
    "26818": "week3",
    "26825": "week4",
    "26AUG": "monthly"
}

# 3. Locate all local Parquet files (excluding /content/drive)
all_local_parquets = []
for root, dirs, files in os.walk("/content"):
    if "/content/drive" in root:
        continue
    for f in files:
        if f.endswith(".parquet"):
            all_local_parquets.append(os.path.join(root, f))

print(f"🔍 Found {len(all_local_parquets)} local Parquet files to move.")

options_moved = 0
spot_moved = 0
skipped_count = 0

# 4. Move files directly into existing structure
for file_path in all_local_parquets:
    file_name = os.path.basename(file_path)

    # A. Move Spot Data
    if any(k in file_name for k in ["NIFTY50", "SPOT", "INDEX"]):
        dest_dir = DRIVE_SPOT_DIR
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, file_name)

        if os.path.exists(dest_path):
            skipped_count += 1
            print(f"⚡ Skipped (already exists): {file_name}")
        else:
            shutil.move(file_path, dest_path)
            spot_moved += 1
            print(f"📈 Moved Spot ➔ {file_name}")
        continue

    # B. Move August Options Data into matching existing subfolders
    for code, subfolder in code_to_folder.items():
        if code in file_name:
            dest_dir = os.path.join(DRIVE_OPTIONS_AUG, subfolder)
            os.makedirs(dest_dir, exist_ok=True)
            dest_path = os.path.join(dest_dir, file_name)

            if os.path.exists(dest_path):
                skipped_count += 1
                print(f"⚡ Skipped (already exists): {file_name}")
            else:
                shutil.move(file_path, dest_path)
                options_moved += 1
                print(f"📦 Moved Option ➔ {file_name} ➔ /{subfolder}/")
            break

try:
    os.sync()
except AttributeError:
    pass

print(f"\n🎉 Transfer Complete!")
print(f"• Option files moved: {options_moved}")
print(f"• Spot files moved: {spot_moved}")
print(f"• Existing files skipped: {skipped_count}")

Mounted at /content/drive
🔍 Found 0 local Parquet files to move.

🎉 Transfer Complete!
• Option files moved: 0
• Spot files moved: 0
• Existing files skipped: 0


In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/fyers_backtest_data"

print("🔍 Searching for existing Parquet files in Google Drive...\n")
found_files = []

for root, dirs, files in os.walk(DRIVE_BASE):
    for f in files:
        if f.endswith(".parquet"):
            found_files.append(os.path.join(root, f))

if found_files:
    print(f"✅ Found {len(found_files)} total Parquet files in Drive:")
    for path in found_files[:10]:  # Print first 10 paths
        print(f" - {path}")
    if len(found_files) > 10:
        print(f" ... and {len(found_files) - 10} more.")
else:
    print("❌ No Parquet files found in Google Drive. You will need to run the downloader script for August.")

🔍 Searching for existing Parquet files in Google Drive...

✅ Found 401 total Parquet files in Drive:
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723500CE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723500PE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723550CE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723550PE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723600CE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723600PE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723650CE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723650PE.parquet
 - /content/drive/MyDrive/fyers_backtest_data/options/year=

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/fyers_backtest_data"

print("🔍 Locating all 401 Parquet files across Google Drive...\n")

located_paths = []
for root, dirs, files in os.walk(DRIVE_BASE):
    for f in files:
        if f.endswith(".parquet"):
            full_path = os.path.join(root, f)
            located_paths.append(full_path)

print(f"✅ Found {len(located_paths)} Parquet files.")

# Group by parent folder to see where they actually sit
from collections import Counter
folder_counts = Counter([os.path.dirname(p) for p in located_paths])

print("\n📊 File location breakdown:")
for folder, count in folder_counts.items():
    print(f" • {folder} ➔ {count} files")

Mounted at /content/drive
🔍 Locating all 401 Parquet files across Google Drive...

✅ Found 401 Parquet files.

📊 File location breakdown:
 • /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1 ➔ 100 files
 • /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week2 ➔ 100 files
 • /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week3 ➔ 100 files
 • /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/monthly ➔ 100 files
 • /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=09 ➔ 1 files


In [ ]:
import os
import polars as pl
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/fyers_backtest_data"
JULY_DIR = os.path.join(DRIVE_BASE, "options", "year=2026", "month=07")

print("🔍 Inspecting files inside month=07...\n")

sample_files = []
for root, dirs, files in os.walk(JULY_DIR):
    for f in files:
        if f.endswith(".parquet"):
            sample_files.append(os.path.join(root, f))

# Look at file codes in filenames
jul_count = sum(1 for p in sample_files if "267" in os.path.basename(p) or "26JUL" in os.path.basename(p))
aug_count = sum(1 for p in sample_files if "268" in os.path.basename(p) or "26AUG" in os.path.basename(p))

print(f"📄 Total Parquet files scanned in month=07: {len(sample_files)}")
print(f" • July contract symbols (267 / 26JUL): {jul_count}")
print(f" • August contract symbols (268 / 26AUG): {aug_count}")

# Check internal Parquet metadata from one file
if sample_files:
    test_file = sample_files[0]
    df = pl.read_parquet(test_file)
    print("\n📋 Sample file inspection:")
    print(f" • File Path: {test_file}")
    print(f" • Symbol Column: {df['symbol'][0]}")
    print(f" • Expiry Date Column: {df['expiry_date'][0]}")
    print(f" • First Timestamp: {df['timestamp'].min()}")
    print(f" • Last Timestamp: {df['timestamp'].max()}")

Mounted at /content/drive
🔍 Inspecting files inside month=07...

📄 Total Parquet files scanned in month=07: 400
 • July contract symbols (267 / 26JUL): 400
 • August contract symbols (268 / 26AUG): 0

📋 Sample file inspection:
 • File Path: /content/drive/MyDrive/fyers_backtest_data/options/year=2026/month=07/week1/NIFTY2670723500CE.parquet
 • Symbol Column: NIFTY2670723500CE
 • Expiry Date Column: 2026-07-07
 • First Timestamp: 2026-07-01 09:15:00+05:30
 • Last Timestamp: 2026-07-07 15:29:00+05:30


In [ ]:
import os
import polars as pl
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/fyers_backtest_data"
JULY_OPTIONS_DIR = os.path.join(DRIVE_BASE, "options", "year=2026", "month=07")
JULY_SPOT_DIR = os.path.join(DRIVE_BASE, "spot", "year=2026", "month=07")

# --- 1. SPOT DATA VERIFICATION ---
print("🔍 --- 1. SPOT DATA VERIFICATION ---")
if os.path.exists(JULY_SPOT_DIR):
    spot_files = [os.path.join(JULY_SPOT_DIR, f) for f in os.listdir(JULY_SPOT_DIR) if f.endswith('.parquet')]
    if spot_files:
        df_spot = pl.read_parquet(spot_files[0])
        spot_days = df_spot['timestamp'].dt.date().unique().sort()
        print(f"✅ Spot file found: {len(df_spot)} rows across {len(spot_days)} trading days.")
        print(f"   Date Range: {spot_days.min()} to {spot_days.max()}")
    else:
        print("❌ Spot directory exists but contains 0 parquet files.")
else:
    print("❌ Spot directory for month=07 not found.")

# --- 2. OPTIONS DATA VERIFICATION ---
print("\n🔍 --- 2. OPTIONS DATA VERIFICATION ---")
subfolders = ["week1", "week2", "week3", "monthly"]
expected_trading_days = 23  # Total trading days in July 2026

summary = []

for folder in subfolders:
    folder_path = os.path.join(JULY_OPTIONS_DIR, folder)
    if not os.path.exists(folder_path):
        summary.append({"Folder": folder, "Status": "Missing Directory", "Files": 0, "Avg Rows": 0})
        continue

    files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

    if not files:
        summary.append({"Folder": folder, "Status": "Empty", "Files": 0, "Avg Rows": 0})
        continue

    # Inspect the first file of the batch
    sample_file = os.path.join(folder_path, files[0])
    df_sample = pl.read_parquet(sample_file)
    trading_days_count = len(df_sample['timestamp'].dt.date().unique())

    summary.append({
        "Folder": folder,
        "Files": len(files),
        "Sample Contract": df_sample["symbol"][0],
        "Rows": len(df_sample),
        "Unique Trading Days": trading_days_count,
        "Start Date": df_sample["timestamp"].min().strftime("%Y-%m-%d"),
        "End Date": df_sample["timestamp"].max().strftime("%Y-%m-%d")
    })

# Output Report
summary_df = pl.DataFrame(summary)
print(summary_df)

Mounted at /content/drive
🔍 --- 1. SPOT DATA VERIFICATION ---
❌ Spot directory exists but contains 0 parquet files.

🔍 --- 2. OPTIONS DATA VERIFICATION ---
shape: (4, 7)
┌─────────┬───────┬───────────────────┬──────┬─────────────────────┬────────────┬────────────┐
│ Folder  ┆ Files ┆ Sample Contract   ┆ Rows ┆ Unique Trading Days ┆ Start Date ┆ End Date   │
│ ---     ┆ ---   ┆ ---               ┆ ---  ┆ ---                 ┆ ---        ┆ ---        │
│ str     ┆ i64   ┆ str               ┆ i64  ┆ i64                 ┆ str        ┆ str        │
╞═════════╪═══════╪═══════════════════╪══════╪═════════════════════╪════════════╪════════════╡
│ week1   ┆ 100   ┆ NIFTY2670723500CE ┆ 1875 ┆ 5                   ┆ 2026-07-01 ┆ 2026-07-07 │
│ week2   ┆ 100   ┆ NIFTY2671423500CE ┆ 3750 ┆ 10                  ┆ 2026-07-01 ┆ 2026-07-14 │
│ week3   ┆ 100   ┆ NIFTY2672123500CE ┆ 5225 ┆ 14                  ┆ 2026-07-02 ┆ 2026-07-21 │
│ monthly ┆ 100   ┆ NIFTY26JUL23500CE ┆ 7500 ┆ 20                  ┆ 2

In [ ]:
import os
import time
import requests
import polars as pl
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration & Paths
YEAR = "2026"
BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
SPOT_BASE_DIR = os.path.join(BASE_DIR, "spot", f"year={YEAR}")
SPOT_URL = "https://api-t1.fyers.in/data/history"

# Ensure HEADERS uses your active APP_ID and access_token
HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}

# List of months to download (Jan through Sep 2026)
months_config = [
    {"month": "01", "start": f"{YEAR}-01-01", "end": f"{YEAR}-01-31"},
    {"month": "02", "start": f"{YEAR}-02-01", "end": f"{YEAR}-02-28"},
    {"month": "03", "start": f"{YEAR}-03-01", "end": f"{YEAR}-03-31"},
    {"month": "04", "start": f"{YEAR}-04-01", "end": f"{YEAR}-04-30"},
    {"month": "05", "start": f"{YEAR}-05-01", "end": f"{YEAR}-05-31"},
    {"month": "06", "start": f"{YEAR}-06-01", "end": f"{YEAR}-06-30"},
    {"month": "07", "start": f"{YEAR}-07-01", "end": f"{YEAR}-07-31"},
    {"month": "08", "start": f"{YEAR}-08-01", "end": f"{YEAR}-08-31"},
    {"month": "09", "start": f"{YEAR}-09-01", "end": f"{YEAR}-09-30"}
]

print("⚡ Starting bulk download of Nifty Spot 1-min candles (Jan 2026 – Sep 2026)...\n")

saved_count = 0
skipped_count = 0

for item in months_config:
    month_str = item["month"]
    start_date = item["start"]
    end_date = item["end"]

    # Hive Directory: fyers_backtest_data/spot/year=2026/month={MM}/
    target_dir = os.path.join(SPOT_BASE_DIR, f"month={month_str}")
    os.makedirs(target_dir, exist_ok=True)

    spot_file = os.path.join(target_dir, "NIFTY50.parquet")

    # Skip if file already exists in Drive
    if os.path.exists(spot_file) and os.path.getsize(spot_file) > 0:
        print(f"⚡ month={month_str} | File already exists. Skipping.")
        skipped_count += 1
        continue

    params = {
        "symbol": "NSE:NIFTY50-INDEX",
        "resolution": "1",
        "date_format": "1",
        "range_from": start_date,
        "range_to": end_date,
        "cont_flag": "1"
    }

    try:
        res = requests.get(SPOT_URL, headers=HEADERS, params=params).json()

        if res.get("s") == "ok" and res.get("candles"):
            df_spot = pl.DataFrame(
                res["candles"],
                schema=["epoch", "open", "high", "low", "close", "volume"],
                orient="row"
            )

            df_spot = df_spot.with_columns([
                pl.from_epoch("epoch", time_unit="s")
                  .dt.replace_time_zone("UTC")
                  .dt.convert_time_zone("Asia/Kolkata")
                  .alias("timestamp"),
                (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
                (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
                (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
                (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32"),
                pl.lit("NIFTY50").cast(pl.Categorical).alias("symbol")
            ]).select(["timestamp", "symbol", "open_i32", "high_i32", "low_i32", "close_i32", "volume"])

            df_spot.write_parquet(spot_file, compression="zstd", compression_level=3)

            min_p = df_spot["close_i32"].min() / 100.0
            max_p = df_spot["close_i32"].max() / 100.0
            saved_count += 1
            print(f"✅ month={month_str} | Saved {len(df_spot)} candles (Range: {min_p:.2f} - {max_p:.2f}) ➔ {spot_file}")
        else:
            print(f"❌ month={month_str} | Error fetching data: {res}")

    except Exception as e:
        print(f"❌ month={month_str} | Exception occurred: {e}")

    time.sleep(0.15)  # Rate-limiting pause

try:
    os.sync()
except AttributeError:
    pass

print(f"\n🎉 Spot Data Collection Complete!")
print(f"• Saved: {saved_count} months")
print(f"• Skipped: {skipped_count} months")

Mounted at /content/drive
⚡ Starting bulk download of Nifty Spot 1-min candles (Jan 2026 – Sep 2026)...

✅ month=01 | Saved 7500 candles (Range: 24923.00 - 26371.05) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=01/NIFTY50.parquet
✅ month=02 | Saved 7875 candles (Range: 24669.00 - 25994.60) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=02/NIFTY50.parquet
✅ month=03 | Saved 7125 candles (Range: 22291.20 - 24978.85) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=03/NIFTY50.parquet
✅ month=04 | Saved 7500 candles (Range: 22186.35 - 24598.35) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=04/NIFTY50.parquet
✅ month=05 | Saved 7125 candles (Range: 23268.50 - 24468.00) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=05/NIFTY50.parquet
✅ month=06 | Saved 7875 candles (Range: 23079.70 - 24255.50) ➔ /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=06/NIFTY50.parquet
✅ month=07 | Saved 

In [ ]:
import requests

url = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
headers = {"Authorization": f"{APP_ID}:{access_token}"}

params = {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": "2026-01-01",
    "range_to": "2026-01-31",
    "date_format": "1"
}

res = requests.get(url, headers=headers, params=params).json()

print("🔍 FYERS Expired Expiry Dates Response:")
print(res)

🔍 FYERS Expired Expiry Dates Response:
{'code': 200, 'data': {'symbol': 'NIFTY', 'from_date': '2026-01-01', 'to_date': '2026-01-31', 'expiry_dates': {'futures': ['2026-01-27'], 'options': ['2026-01-06', '2026-01-13', '2026-01-20', '2026-01-27']}}, 'message': '', 's': 'ok'}


In [ ]:
import requests

EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

headers = {"Authorization": f"{APP_ID}:{access_token}"}

# Test Jan 2026 option contract (January 6, 2026 weekly expiry)
params = {
    "symbol": "NSE:NIFTY2610625000CE",
    "resolution": "1",
    "date_format": "1",
    "range_from": "2026-01-01",
    "range_to": "2026-01-01"
}

res = requests.get(EXPIRED_CANDLES_URL, headers=headers, params=params).json()

print("🔍 Diagnostic Output for Jan 2026 Expired Option Candles:")
print(f"Status: {res.get('s')} | Message: {res.get('message', '')}")

candles = res.get("candles") or res.get("data", {}).get("candles", [])
print(f"Candles Found: {len(candles)}")
if candles:
    print(f"Sample Candle: {candles[0]}")

🔍 Diagnostic Output for Jan 2026 Expired Option Candles:
Status: ok | Message: 
Candles Found: 374
Sample Candle: [1767239160, 1183.95, 1207.0, 1183.95, 1207.0, 716]


spot data download in proper folder

In [ ]:
import os
import requests
import polars as pl
from datetime import date, timedelta
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# PIPELINE RANGE CONFIGURATION
# Set target range (e.g. Oct 2025 to Sep 2026)
# ----------------------------------------------------------------------
START_YEAR, START_MONTH = 2025, 10
END_YEAR,   END_MONTH   = 2026, 9

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
SPOT_BASE_DIR = os.path.join(BASE_DIR, "spot")

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
spot_url = "https://api-t1.fyers.in/data/history"

def download_spot_month(yr, mo):
    year_str = f"{yr:04d}"
    month_str = f"{mo:02d}"

    # Target existing Hive folder: /fyers_backtest_data/spot/year=YYYY/month=MM/
    spot_dir = os.path.join(SPOT_BASE_DIR, f"year={year_str}", f"month={month_str}")

    # Create folder ONLY if it doesn't already exist
    os.makedirs(spot_dir, exist_ok=True)
    spot_file = os.path.join(spot_dir, "NIFTY50.parquet")

    # SKIP LOGIC: If file exists and is not empty, skip network call completely
    if os.path.exists(spot_file) and os.path.getsize(spot_file) > 0:
        print(f"⏩ Spot data already exists at {spot_file}. Skipping {year_str}-{month_str}.")
        return

    # Calculate exact start and end date for the target month
    start_date = f"{year_str}-{month_str}-01"
    last_day = 31 if mo == 12 else (date(yr, mo + 1, 1) - timedelta(days=1)).day
    end_date = f"{year_str}-{month_str}-{last_day:02d}"

    params = {
        "symbol": "NSE:NIFTY50-INDEX",
        "resolution": "1",
        "date_format": "1",
        "range_from": start_date,
        "range_to": end_date,
        "cont_flag": "1"
    }

    print(f"⚡ Downloading Nifty Spot 1-min candles for {year_str}-{month_str}...")
    try:
        res_obj = requests.get(spot_url, headers=HEADERS, params=params, timeout=15)
        res = res_obj.json()
    except Exception as e:
        print(f"❌ HTTP Error for {year_str}-{month_str}: {e}")
        return

    if res.get("s") == "ok" and res.get("candles"):
        df_spot = pl.DataFrame(
            res["candles"],
            schema=["epoch", "open", "high", "low", "close", "volume"],
            orient="row"
        )

        df_spot = df_spot.with_columns([
            pl.from_epoch("epoch", time_unit="s")
              .dt.replace_time_zone("UTC")
              .dt.convert_time_zone("Asia/Kolkata")
              .alias("timestamp"),
            (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
            (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
            (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
            (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32"),
            pl.lit("NIFTY50").cast(pl.Categorical).alias("symbol")
        ]).select(["timestamp", "symbol", "open_i32", "high_i32", "low_i32", "close_i32", "volume"])

        df_spot.write_parquet(spot_file, compression="zstd", compression_level=3)

        min_p = df_spot["close_i32"].min() / 100.0
        max_p = df_spot["close_i32"].max() / 100.0
        print(f"✅ Saved Nifty Spot data ({len(df_spot):,} candles) to: {spot_file}")
        print(f"📈 Spot Range: Min = {min_p:.2f}, Max = {max_p:.2f}")
    else:
        print(f"❌ Error fetching spot data for {year_str}-{month_str}: {res}")

# Execution loop across target months
curr_y, curr_m = START_YEAR, START_MONTH
while (curr_y < END_YEAR) or (curr_y == END_YEAR and curr_m <= END_MONTH):
    download_spot_month(curr_y, curr_m)
    curr_m += 1
    if curr_m > 12:
        curr_m = 1
        curr_y += 1

print("\n🎉 Spot Data Pipeline Completed Successfully!")

Mounted at /content/drive
⚡ Downloading Nifty Spot 1-min candles for 2025-10...
✅ Saved Nifty Spot data (7,561 candles) to: /content/drive/MyDrive/fyers_backtest_data/spot/year=2025/month=10/NIFTY50.parquet
📈 Spot Range: Min = 24619.85, Max = 26103.50
⚡ Downloading Nifty Spot 1-min candles for 2025-11...
✅ Saved Nifty Spot data (7,125 candles) to: /content/drive/MyDrive/fyers_backtest_data/spot/year=2025/month=11/NIFTY50.parquet
📈 Spot Range: Min = 25320.20, Max = 26308.40
⚡ Downloading Nifty Spot 1-min candles for 2025-12...
✅ Saved Nifty Spot data (8,250 candles) to: /content/drive/MyDrive/fyers_backtest_data/spot/year=2025/month=12/NIFTY50.parquet
📈 Spot Range: Min = 25699.40, Max = 26311.00
⏩ Spot data already exists at /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=01/NIFTY50.parquet. Skipping 2026-01.
⏩ Spot data already exists at /content/drive/MyDrive/fyers_backtest_data/spot/year=2026/month=02/NIFTY50.parquet. Skipping 2026-02.
⏩ Spot data already exists at /c

one moth complete data download

In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# SINGLE MONTH CONFIGURATION
# ----------------------------------------------------------------------
TARGET_YEAR  = 2025
TARGET_MONTH = 10

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Ultra-safe worker thread count for maximum completion rate

def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Skip already downloaded and non-empty files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:  # Rate limited -> backing off
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)  # Micro-pace requests to avoid hitting rate limits
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# EXECUTION PIPELINE
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Run Stage 1 spot downloader first!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-31",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            # Robust ATM calculation with fallback
            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2025-10
⚡ WEEK1 (2025-10-07) Finished ➔ Saved: 184 | Skipped: 64 | NoData: 0 | Errors: 0
⚡ WEEK2 (2025-10-14) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2025-10-20) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2025-10-28) Finished ➔ Saved: 276 | Skipped: 32 | NoData: 2 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2025-10!


for nov25

In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR NOVEMBER 2025)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2025
TARGET_MONTH = 11

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Paced execution for 100% data integrity without HTTP 429 rate limits

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff on rate-limiting."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR NOVEMBER 2025
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Nov 2025!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-30",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2025-11
⚡ WEEK1 (2025-11-04) Finished ➔ Saved: 308 | Skipped: 0 | NoData: 0 | Errors: 2
⚡ WEEK2 (2025-11-11) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2025-11-18) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2025-11-25) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2025-11!


for dec 2026

In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR DECEMBER 2025)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2025
TARGET_MONTH = 12

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR DECEMBER 2025
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Dec 2025!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-31",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2025-12
⚡ WEEK1 (2025-12-02) Finished ➔ Saved: 124 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK2 (2025-12-09) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2025-12-16) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK4 (2025-12-23) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2025-12-30) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2025-12!


In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR JANUARY 2026)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2026
TARGET_MONTH = 1

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR JANUARY 2026
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Jan 2026!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-31",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2026-01
⚡ WEEK1 (2026-01-06) Finished ➔ Saved: 62 | Skipped: 248 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-01-13) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-01-20) Finished ➔ Saved: 0 | Skipped: 248 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-01-27) Finished ➔ Saved: 51 | Skipped: 197 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2026-01!


In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR FEBRUARY 2026)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2026
TARGET_MONTH = 2

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR FEBRUARY 2026
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Feb 2026!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-28",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2026-02
⚡ WEEK1 (2026-02-03) Finished ➔ Saved: 372 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-02-10) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-02-17) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-02-24) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2026-02!


In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR MARCH 2026)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2026
TARGET_MONTH = 3

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR MARCH 2026
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Mar 2026!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-31",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2026-03
⚡ WEEK1 (2026-03-02) Finished ➔ Saved: 62 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-03-10) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-03-17) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK4 (2026-03-24) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-03-30) Finished ➔ Saved: 186 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2026-03!


In [ ]:
import os
import glob
import polars as pl

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
MAR_DIR = os.path.join(BASE_DIR, "options", "year=2026", "month=03")

print("🔍 STARTING COMPREHENSIVE AUDIT & BREAKDOWN FOR MARCH 2026\n" + "="*75)

folders = ["week1", "week2", "week3", "week4", "monthly"]
total_files = 0
total_size = 0
corrupt_files = []
null_value_files = []

for folder in folders:
    fpath = os.path.join(MAR_DIR, folder)
    if not os.path.exists(fpath):
        continue

    files = glob.glob(os.path.join(fpath, "*.parquet"))
    if not files:
        continue

    total_files += len(files)
    fsize = sum(os.path.getsize(f) for f in files)
    total_size += fsize

    min_date, max_date = None, None

    for f in files:
        try:
            df = pl.read_parquet(f)
            if df.null_count().to_dicts()[0] != {col: 0 for col in df.columns}:
                null_value_files.append(f)

            if not df.is_empty():
                fdates = df["timestamp"].dt.date()
                fmin, fmax = fdates.min(), fdates.max()
                if min_date is None or fmin < min_date: min_date = fmin
                if max_date is None or fmax > max_date: max_date = fmax
        except Exception as e:
            corrupt_files.append((f, str(e)))

    print(f"📂 Folder: {folder.upper():<7} | Files: {len(files):<4} | Size: {fsize / (1024*1024):.2f} MB | Date Range: {min_date} ➔ {max_date}")

print("\n" + "="*75)
print(f"📊 SUMMARY REPORT:")
print(f"  • Total Parquet Files Saved : {total_files}")
print(f"  • Total Storage Footprint   : {total_size / (1024*1024):.2f} MB")
print(f"  • Corrupt / Unreadable Files: {len(corrupt_files)}")
print(f"  • Files with Null Values    : {len(null_value_files)}")

🔍 STARTING COMPREHENSIVE AUDIT & BREAKDOWN FOR MARCH 2026
📂 Folder: WEEK1   | Files: 62   | Size: 0.22 MB | Date Range: 2026-03-02 ➔ 2026-03-02
📂 Folder: WEEK2   | Files: 310  | Size: 1.15 MB | Date Range: 2026-03-04 ➔ 2026-03-10
📂 Folder: WEEK3   | Files: 310  | Size: 1.17 MB | Date Range: 2026-03-11 ➔ 2026-03-17
📂 Folder: WEEK4   | Files: 310  | Size: 1.17 MB | Date Range: 2026-03-18 ➔ 2026-03-24
📂 Folder: MONTHLY | Files: 186  | Size: 0.70 MB | Date Range: 2026-03-25 ➔ 2026-03-30

📊 SUMMARY REPORT:
  • Total Parquet Files Saved : 1178
  • Total Storage Footprint   : 4.41 MB
  • Corrupt / Unreadable Files: 0
  • Files with Null Values    : 0


In [ ]:
import os
import re
import time
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (SET FOR APRIL 2026)
# ----------------------------------------------------------------------
TARGET_YEAR  = 2026
TARGET_MONTH = 4

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-31",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN DOWNLOAD PIPELINE FOR APRIL 2026
# ----------------------------------------------------------------------
year_str = f"{TARGET_YEAR:04d}"
month_str = f"{TARGET_MONTH:02d}"
month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}", flush=True)

# Spot Data Dependency Check
spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
if not os.path.exists(spot_curr_path):
    raise FileNotFoundError(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for Apr 2026!")

prev_mo_val = 12 if TARGET_MONTH == 1 else TARGET_MONTH - 1
prev_yr_val = TARGET_YEAR - 1 if TARGET_MONTH == 1 else TARGET_YEAR
spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

if os.path.exists(spot_prev_path):
    df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
else:
    df_spot = pl.read_parquet(spot_curr_path)

df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

# Dynamic Expiry Retrieval
res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
    "symbol": "NSE:NIFTY50-INDEX",
    "range_from": f"{year_str}-{month_str}-01",
    "range_to": f"{year_str}-{month_str}-30",
    "date_format": "1"
})

if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
    api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
    monthly_expiry_date = api_expiries[-1]
    weekly_expiries = api_expiries[:-1]

    # Map subfolders
    folder_map = {}
    for idx, exp_date in enumerate(weekly_expiries):
        folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
    folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

    for folder_path in folder_map.values():
        os.makedirs(folder_path, exist_ok=True)

    prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, TARGET_MONTH)

    # Process each expiry contract cycle
    for exp_date in api_expiries:
        target_folder = folder_map[exp_date]
        folder_label = os.path.basename(target_folder)

        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })

        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        start_dt = pl.lit(prev_expiry).str.to_date()
        end_dt = pl.lit(exp_date).str.to_date()

        trading_dates = df_spot.filter(
            (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
        )["date"].unique().sort()

        if len(trading_dates) == 0:
            prev_expiry = exp_date
            continue

        # ATM ± 15 strike task construction
        download_tasks = []
        for current_date in trading_dates:
            date_str = current_date.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == current_date)
            if day_spot.is_empty():
                continue

            if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                day_open_spot = day_spot["open_i32"][0] / 100.0
            else:
                day_open_spot = day_spot["close_i32"].mean() / 100.0

            day_atm = int(round(day_open_spot / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for symbol_name in raw_options:
                match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                if match and int(match.group(1)) in day_strikes:
                    download_tasks.append((symbol_name, date_str, target_folder))

        # Parallel Worker Execution
        saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_contract_day, sym, d_str, folder)
                for sym, d_str, folder in download_tasks
            ]
            for future in as_completed(futures):
                st = future.result()
                if st == "saved": saved_cnt += 1
                elif st == "skipped": skipped_cnt += 1
                elif st == "no_data": no_data_cnt += 1
                else: error_cnt += 1

        print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
        prev_expiry = exp_date

print(f"\n🎉 Options Download Completed with 100% Integrity for {year_str}-{month_str}!", flush=True)

Mounted at /content/drive
🛡️ High-Accuracy Options Downloader Starting for: 2026-04
⚡ WEEK1 (2026-04-07) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-04-13) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-04-21) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-04-28) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Completed with 100% Integrity for 2026-04!


In [ ]:
import os
import glob
import polars as pl

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
APR_DIR = os.path.join(BASE_DIR, "options", "year=2026", "month=04")

print("🔍 STARTING COMPREHENSIVE AUDIT & BREAKDOWN FOR APRIL 2026\n" + "="*75)

folders = ["week1", "week2", "week3", "week4", "monthly"]
total_files = 0
total_size = 0
corrupt_files = []
null_value_files = []

for folder in folders:
    fpath = os.path.join(APR_DIR, folder)
    if not os.path.exists(fpath):
        continue

    files = glob.glob(os.path.join(fpath, "*.parquet"))
    if not files:
        continue

    total_files += len(files)
    fsize = sum(os.path.getsize(f) for f in files)
    total_size += fsize

    min_date, max_date = None, None

    for f in files:
        try:
            df = pl.read_parquet(f)
            if df.null_count().to_dicts()[0] != {col: 0 for col in df.columns}:
                null_value_files.append(f)

            if not df.is_empty():
                fdates = df["timestamp"].dt.date()
                fmin, fmax = fdates.min(), fdates.max()
                if min_date is None or fmin < min_date: min_date = fmin
                if max_date is None or fmax > max_date: max_date = fmax
        except Exception as e:
            corrupt_files.append((f, str(e)))

    print(f"📂 Folder: {folder.upper():<7} | Files: {len(files):<4} | Size: {fsize / (1024*1024):.2f} MB | Date Range: {min_date} ➔ {max_date}")

print("\n" + "="*75)
print(f"📊 SUMMARY REPORT:")
print(f"  • Total Parquet Files Saved : {total_files}")
print(f"  • Total Storage Footprint   : {total_size / (1024*1024):.2f} MB")
print(f"  • Corrupt / Unreadable Files: {len(corrupt_files)}")
print(f"  • Files with Null Values    : {len(null_value_files)}")

🔍 STARTING COMPREHENSIVE AUDIT & BREAKDOWN FOR APRIL 2026
📂 Folder: WEEK1   | Files: 248  | Size: 0.94 MB | Date Range: 2026-04-01 ➔ 2026-04-07
📂 Folder: WEEK2   | Files: 248  | Size: 0.91 MB | Date Range: 2026-04-08 ➔ 2026-04-13
📂 Folder: WEEK3   | Files: 310  | Size: 1.12 MB | Date Range: 2026-04-15 ➔ 2026-04-21
📂 Folder: MONTHLY | Files: 310  | Size: 1.12 MB | Date Range: 2026-04-22 ➔ 2026-04-28

📊 SUMMARY REPORT:
  • Total Parquet Files Saved : 1116
  • Total Storage Footprint   : 4.09 MB
  • Corrupt / Unreadable Files: 0
  • Files with Null Values    : 0


may , june and july 2026

In [ ]:
import os
import re
import time
import calendar
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS
# ----------------------------------------------------------------------
TARGET_YEAR = 2026
TARGET_MONTHS = [5, 6, 7]  # May, June, July

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr
    last_day_prev = calendar.monthrange(prev_yr, prev_mo)[1]

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-{last_day_prev:02d}",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN BATCH DOWNLOAD PIPELINE
# ----------------------------------------------------------------------
for target_month in TARGET_MONTHS:
    year_str = f"{TARGET_YEAR:04d}"
    month_str = f"{target_month:02d}"
    month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

    # Calculate exact last day of target month (28, 30, or 31)
    last_day_month = calendar.monthrange(TARGET_YEAR, target_month)[1]

    print(f"\n===========================================================================")
    print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}")
    print(f"===========================================================================")

    # Spot Data Dependency Check
    spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
    if not os.path.exists(spot_curr_path):
        print(f"⚠️ Spot data missing at {spot_curr_path}. Skipping month {month_str}!")
        continue

    prev_mo_val = 12 if target_month == 1 else target_month - 1
    prev_yr_val = TARGET_YEAR - 1 if target_month == 1 else TARGET_YEAR
    spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

    if os.path.exists(spot_prev_path):
        df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
    else:
        df_spot = pl.read_parquet(spot_curr_path)

    df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

    # Dynamic Expiry Retrieval with Correct Month-End Date
    res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{year_str}-{month_str}-01",
        "range_to": f"{year_str}-{month_str}-{last_day_month:02d}",
        "date_format": "1"
    })

    if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
        api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
        monthly_expiry_date = api_expiries[-1]
        weekly_expiries = api_expiries[:-1]

        # Map subfolders
        folder_map = {}
        for idx, exp_date in enumerate(weekly_expiries):
            folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
        folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

        for folder_path in folder_map.values():
            os.makedirs(folder_path, exist_ok=True)

        prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, target_month)

        # Process each expiry contract cycle
        for exp_date in api_expiries:
            target_folder = folder_map[exp_date]
            folder_label = os.path.basename(target_folder)

            sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
                "symbol": "NSE:NIFTY50-INDEX",
                "expiry_date": exp_date
            })

            raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

            start_dt = pl.lit(prev_expiry).str.to_date()
            end_dt = pl.lit(exp_date).str.to_date()

            trading_dates = df_spot.filter(
                (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
            )["date"].unique().sort()

            if len(trading_dates) == 0:
                prev_expiry = exp_date
                continue

            # ATM ± 15 strike task construction
            download_tasks = []
            for current_date in trading_dates:
                date_str = current_date.strftime("%Y-%m-%d")
                day_spot = df_spot.filter(pl.col("date") == current_date)
                if day_spot.is_empty():
                    continue

                if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                    day_open_spot = day_spot["open_i32"][0] / 100.0
                else:
                    day_open_spot = day_spot["close_i32"].mean() / 100.0

                day_atm = int(round(day_open_spot / 50.0) * 50)
                day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

                for symbol_name in raw_options:
                    match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                    if match and int(match.group(1)) in day_strikes:
                        download_tasks.append((symbol_name, date_str, target_folder))

            # Parallel Worker Execution
            saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(process_single_contract_day, sym, d_str, folder)
                    for sym, d_str, folder in download_tasks
                ]
                for future in as_completed(futures):
                    st = future.result()
                    if st == "saved": saved_cnt += 1
                    elif st == "skipped": skipped_cnt += 1
                    elif st == "no_data": no_data_cnt += 1
                    else: error_cnt += 1

            print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
            prev_expiry = exp_date
    else:
        print(f"❌ Failed to fetch expiry dates for month {month_str}. Verify spot data and API access.")

print(f"\n🎉 Options Download Execution Finished!", flush=True)

Mounted at /content/drive

🛡️ High-Accuracy Options Downloader Starting for: 2026-05
⚡ WEEK1 (2026-05-05) Finished ➔ Saved: 124 | Skipped: 124 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-05-12) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-05-19) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-05-26) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0

🛡️ High-Accuracy Options Downloader Starting for: 2026-06
⚡ WEEK1 (2026-06-02) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK2 (2026-06-09) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-06-16) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK4 (2026-06-23) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-06-30) Finished ➔ Saved: 248 | Skipped: 0 | NoData: 0 | Errors: 0

🛡️ High-Accuracy Options Downloader Starting for: 2026-07
⚡ WEEK1 (2026-07-07) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | E

aug and sep 2026

In [ ]:
import os
import re
import time
import calendar
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS (AUGUST & SEPTEMBER 2026)
# ----------------------------------------------------------------------
TARGET_YEAR = 2026
TARGET_MONTHS = [8, 9]  # August, September

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    """Fetches JSON response from FYERS API with exponential backoff."""
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    """Downloads 1-min option candles, applies scaling/types, and saves to Parquet."""
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip already downloaded valid files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    """Gets previous month's final expiry date to set strict boundary start."""
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr
    last_day_prev = calendar.monthrange(prev_yr, prev_mo)[1]

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-{last_day_prev:02d}",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. MAIN BATCH DOWNLOAD PIPELINE (AUGUST & SEPTEMBER 2026)
# ----------------------------------------------------------------------
for target_month in TARGET_MONTHS:
    year_str = f"{TARGET_YEAR:04d}"
    month_str = f"{target_month:02d}"
    month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")

    # Dynamic month-end (31 days for Aug, 30 days for Sep)
    last_day_month = calendar.monthrange(TARGET_YEAR, target_month)[1]

    print(f"\n===========================================================================")
    print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}")
    print(f"===========================================================================")

    # Spot Data Dependency Check
    spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
    if not os.path.exists(spot_curr_path):
        print(f"⚠️ Spot data missing at {spot_curr_path}. Ensure Stage 1 spot downloader ran for {year_str}-{month_str}!")
        continue

    prev_mo_val = 12 if target_month == 1 else target_month - 1
    prev_yr_val = TARGET_YEAR - 1 if target_month == 1 else TARGET_YEAR
    spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

    if os.path.exists(spot_prev_path):
        df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
    else:
        df_spot = pl.read_parquet(spot_curr_path)

    df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

    # Dynamic Expiry Retrieval
    res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{year_str}-{month_str}-01",
        "range_to": f"{year_str}-{month_str}-{last_day_month:02d}",
        "date_format": "1"
    })

    if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
        api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
        monthly_expiry_date = api_expiries[-1]
        weekly_expiries = api_expiries[:-1]

        # Map subfolders
        folder_map = {}
        for idx, exp_date in enumerate(weekly_expiries):
            folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
        folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

        for folder_path in folder_map.values():
            os.makedirs(folder_path, exist_ok=True)

        prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, target_month)

        # Process each expiry contract cycle
        for exp_date in api_expiries:
            target_folder = folder_map[exp_date]
            folder_label = os.path.basename(target_folder)

            sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
                "symbol": "NSE:NIFTY50-INDEX",
                "expiry_date": exp_date
            })

            raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

            start_dt = pl.lit(prev_expiry).str.to_date()
            end_dt = pl.lit(exp_date).str.to_date()

            trading_dates = df_spot.filter(
                (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
            )["date"].unique().sort()

            if len(trading_dates) == 0:
                prev_expiry = exp_date
                continue

            # ATM ± 15 strike task construction
            download_tasks = []
            for current_date in trading_dates:
                date_str = current_date.strftime("%Y-%m-%d")
                day_spot = df_spot.filter(pl.col("date") == current_date)
                if day_spot.is_empty():
                    continue

                if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                    day_open_spot = day_spot["open_i32"][0] / 100.0
                else:
                    day_open_spot = day_spot["close_i32"].mean() / 100.0

                day_atm = int(round(day_open_spot / 50.0) * 50)
                day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

                for symbol_name in raw_options:
                    match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                    if match and int(match.group(1)) in day_strikes:
                        download_tasks.append((symbol_name, date_str, target_folder))

            # Parallel Worker Execution
            saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(process_single_contract_day, sym, d_str, folder)
                    for sym, d_str, folder in download_tasks
                ]
                for future in as_completed(futures):
                    st = future.result()
                    if st == "saved": saved_cnt += 1
                    elif st == "skipped": skipped_cnt += 1
                    elif st == "no_data": no_data_cnt += 1
                    else: error_cnt += 1

            print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
            prev_expiry = exp_date
    else:
        print(f"❌ Failed to fetch expiry dates for month {month_str}. Verify spot data and API access.")

print(f"\n🎉 Options Download Execution Finished for August & September 2026!", flush=True)

Mounted at /content/drive

🛡️ High-Accuracy Options Downloader Starting for: 2026-08
⚡ WEEK1 (2026-08-04) Finished ➔ Saved: 309 | Skipped: 0 | NoData: 1 | Errors: 0
⚡ WEEK2 (2026-08-11) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-08-18) Finished ➔ Saved: 309 | Skipped: 0 | NoData: 1 | Errors: 0
⚡ MONTHLY (2026-08-25) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0

🛡️ High-Accuracy Options Downloader Starting for: 2026-09
⚡ WEEK1 (2026-09-01) Finished ➔ Saved: 308 | Skipped: 0 | NoData: 2 | Errors: 0
⚡ WEEK2 (2026-09-08) Finished ➔ Saved: 308 | Skipped: 0 | NoData: 2 | Errors: 0
⚡ WEEK3 (2026-09-15) Finished ➔ Saved: 245 | Skipped: 0 | NoData: 3 | Errors: 0
⚡ WEEK4 (2026-09-22) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-09-29) Finished ➔ Saved: 310 | Skipped: 0 | NoData: 0 | Errors: 0

🎉 Options Download Execution Finished for August & September 2026!


In [ ]:
import os
import re
import time
import calendar
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS
# ----------------------------------------------------------------------
TARGET_YEAR = 2026
TARGET_MONTHS = [8, 9]  # August and September

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2  # Rate-limit safety pace

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=8, delay=1.5):
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=10)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(delay)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=10):
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    # Idempotency check: skip existing non-empty files
    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=10)

            if res_obj.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                time.sleep(0.15)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            time.sleep(1.0)

    return "error"

def get_previous_monthly_expiry(yr, mo):
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr
    last_day_prev = calendar.monthrange(prev_yr, prev_mo)[1]

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-{last_day_prev:02d}",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. REVISED TARGETED RECOVERY & DOWNLOADER
# ----------------------------------------------------------------------
for target_month in TARGET_MONTHS:
    year_str = f"{TARGET_YEAR:04d}"
    month_str = f"{target_month:02d}"
    month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")
    last_day_month = calendar.monthrange(TARGET_YEAR, target_month)[1]

    print(f"\n===========================================================================")
    print(f"🛡️ Smart Missing Contract Downloader for: {year_str}-{month_str}")
    print(f"===========================================================================")

    spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
    if not os.path.exists(spot_curr_path):
        print(f"⚠️ Spot data missing at {spot_curr_path}!")
        continue

    prev_mo_val = 12 if target_month == 1 else target_month - 1
    prev_yr_val = TARGET_YEAR - 1 if target_month == 1 else TARGET_YEAR
    spot_prev_path = os.path.join(BASE_DIR, "spot", f"year={prev_yr_val:04d}", f"month={prev_mo_val:02d}", "NIFTY50.parquet")

    if os.path.exists(spot_prev_path):
        df_spot = pl.concat([pl.read_parquet(spot_prev_path), pl.read_parquet(spot_curr_path)])
    else:
        df_spot = pl.read_parquet(spot_curr_path)

    df_spot = df_spot.with_columns(pl.col("timestamp").dt.date().alias("date"))

    res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{year_str}-{month_str}-01",
        "range_to": f"{year_str}-{month_str}-{last_day_month:02d}",
        "date_format": "1"
    })

    if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
        api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
        monthly_expiry_date = api_expiries[-1]
        weekly_expiries = api_expiries[:-1]

        folder_map = {}
        for idx, exp_date in enumerate(weekly_expiries):
            folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
        folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

        for folder_path in folder_map.values():
            os.makedirs(folder_path, exist_ok=True)

        prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, target_month)

        for exp_date in api_expiries:
            target_folder = folder_map[exp_date]
            folder_label = os.path.basename(target_folder)

            sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
                "symbol": "NSE:NIFTY50-INDEX",
                "expiry_date": exp_date
            })

            raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

            start_dt = pl.lit(prev_expiry).str.to_date()
            end_dt = pl.lit(exp_date).str.to_date()

            trading_dates = df_spot.filter(
                (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
            )["date"].unique().sort()

            if len(trading_dates) == 0:
                prev_expiry = exp_date
                continue

            download_tasks = []
            for current_date in trading_dates:
                date_str = current_date.strftime("%Y-%m-%d")
                day_spot = df_spot.filter(pl.col("date") == current_date)
                if day_spot.is_empty():
                    continue

                if "open_i32" in day_spot.columns and not day_spot["open_i32"].is_null().all():
                    day_open_spot = day_spot["open_i32"][0] / 100.0
                else:
                    day_open_spot = day_spot["close_i32"].mean() / 100.0

                day_atm = int(round(day_open_spot / 50.0) * 50)
                day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

                for symbol_name in raw_options:
                    match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                    if match and int(match.group(1)) in day_strikes:
                        download_tasks.append((symbol_name, date_str, target_folder))

            saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(process_single_contract_day, sym, d_str, folder)
                    for sym, d_str, folder in download_tasks
                ]
                for future in as_completed(futures):
                    st = future.result()
                    if st == "saved": saved_cnt += 1
                    elif st == "skipped": skipped_cnt += 1
                    elif st == "no_data": no_data_cnt += 1
                    else: error_cnt += 1

            print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
            prev_expiry = exp_date

print(f"\n🎉 Process Complete!", flush=True)

Mounted at /content/drive

🛡️ Smart Missing Contract Downloader for: 2026-08
⚡ WEEK1 (2026-08-04) Finished ➔ Saved: 0 | Skipped: 309 | NoData: 1 | Errors: 0
⚡ WEEK2 (2026-08-11) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0
⚡ WEEK3 (2026-08-18) Finished ➔ Saved: 0 | Skipped: 309 | NoData: 1 | Errors: 0
⚡ MONTHLY (2026-08-25) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0

🛡️ Smart Missing Contract Downloader for: 2026-09
⚡ WEEK1 (2026-09-01) Finished ➔ Saved: 0 | Skipped: 308 | NoData: 2 | Errors: 0
⚡ WEEK2 (2026-09-08) Finished ➔ Saved: 0 | Skipped: 308 | NoData: 2 | Errors: 0
⚡ WEEK3 (2026-09-15) Finished ➔ Saved: 0 | Skipped: 245 | NoData: 3 | Errors: 0
⚡ WEEK4 (2026-09-22) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0
⚡ MONTHLY (2026-09-29) Finished ➔ Saved: 0 | Skipped: 310 | NoData: 0 | Errors: 0

🎉 Process Complete!


In [ ]:
import os
import glob
import polars as pl
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_DIR = os.path.join(BASE_DIR, "options", "year=2026")

print("🔍 ACCURATE EXPIRY-AWARE AUDIT FOR AUGUST & SEPTEMBER 2026\n" + "="*75)

for month in [8, 9]:
    mstr = f"{month:02d}"
    mpath = os.path.join(OPTIONS_DIR, f"month={mstr}")

    if not os.path.exists(mpath):
        print(f"❌ Path missing: {mpath}")
        continue

    all_files = glob.glob(os.path.join(mpath, "**", "*.parquet"), recursive=True)
    total_size = sum(os.path.getsize(f) for f in all_files)

    print(f"\n🗓️ MONTH 2026-{mstr}:")
    print(f"  • Total Parquet Files Saved : {len(all_files)}")
    print(f"  • Total Storage Size        : {total_size / (1024*1024):.2f} MB")

    # Folder Breakdown
    folders = [d for d in os.listdir(mpath) if os.path.isdir(os.path.join(mpath, d))]
    for folder in sorted(folders):
        f_files = glob.glob(os.path.join(mpath, folder, "*.parquet"))
        f_size = sum(os.path.getsize(f) for f in f_files)
        print(f"   └── Folder: {folder.upper():<8} | Files: {len(f_files):<4} | Size: {f_size / (1024*1024):.2f} MB")

Mounted at /content/drive
🔍 ACCURATE EXPIRY-AWARE AUDIT FOR AUGUST & SEPTEMBER 2026

🗓️ MONTH 2026-08:
  • Total Parquet Files Saved : 1238
  • Total Storage Size        : 4.29 MB
   └── Folder: MONTHLY  | Files: 310  | Size: 1.06 MB
   └── Folder: WEEK1    | Files: 309  | Size: 1.07 MB
   └── Folder: WEEK2    | Files: 310  | Size: 1.09 MB
   └── Folder: WEEK3    | Files: 309  | Size: 1.08 MB
   └── Folder: WEEK4    | Files: 0    | Size: 0.00 MB

🗓️ MONTH 2026-09:
  • Total Parquet Files Saved : 1481
  • Total Storage Size        : 5.22 MB
   └── Folder: MONTHLY  | Files: 310  | Size: 1.11 MB
   └── Folder: WEEK1    | Files: 308  | Size: 1.08 MB
   └── Folder: WEEK2    | Files: 308  | Size: 1.07 MB
   └── Folder: WEEK3    | Files: 245  | Size: 0.87 MB
   └── Folder: WEEK4    | Files: 310  | Size: 1.09 MB


In [ ]:
import calendar
import re
import requests
import polars as pl
import os

# Query FYERS API to identify NoData contracts and calculate distance from ATM
print("🔍 AUDITING EXACT 'NODATA' CONTRACTS AND DISTANCE FROM ATM\n" + "=" * 70)

for month in [8, 9]:
    month_str = f"{month:02d}"
    last_day = calendar.monthrange(2026, month)[1]

    # 1. Fetch Expiring dates
    res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"2026-{month_str}-01",
        "range_to": f"2026-{month_str}-{last_day:02d}",
        "date_format": "1"
    })

    expiries = res_exp.get("data", {}).get("expiry_dates", {}).get("options", [])

    # 2. Load Spot Data
    spot_path = os.path.join(BASE_DIR, "spot", "year=2026", f"month={month_str}", "NIFTY50.parquet")
    if not os.path.exists(spot_path): continue
    df_spot = pl.read_parquet(spot_path).with_columns(pl.col("timestamp").dt.date().alias("date"))

    for exp_date in expiries:
        sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
            "symbol": "NSE:NIFTY50-INDEX",
            "expiry_date": exp_date
        })
        raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

        for d in df_spot["date"].unique().sort():
            d_str = d.strftime("%Y-%m-%d")
            day_spot = df_spot.filter(pl.col("date") == d)
            if day_spot.is_empty(): continue

            day_open = day_spot["open_i32"][0] / 100.0 if "open_i32" in day_spot.columns else day_spot["close_i32"].mean() / 100.0
            day_atm = int(round(day_open / 50.0) * 50)
            day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

            for sym in raw_options:
                m = re.search(r"(\d{5})(CE|PE)$", sym)
                if m:
                    strike_val = int(m.group(1))
                    opt_type = m.group(2)
                    if strike_val in day_strikes:
                        params = {"symbol": sym, "resolution": "1", "date_format": "1", "range_from": d_str, "range_to": d_str}
                        res = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params).json()

                        if res.get("s") == "no_data":
                            strike_diff_pts = strike_val - day_atm
                            strike_diff_steps = int(strike_diff_pts / 50)
                            print(f"📌 Date: {d_str} | Expiry: {exp_date}")
                            print(f"   • Symbol      : {sym}")
                            print(f"   • ATM Spot    : {day_atm}")
                            print(f"   • Strike      : {strike_val} {opt_type}")
                            print(f"   • ATM Offset  : {strike_diff_steps:+d} strikes ({strike_diff_pts:+d} pts from ATM)\n")

Streaming output truncated to the last 5000 lines.
   • ATM Offset  : -15 strikes (-750 pts from ATM)

📌 Date: 2026-09-24 | Expiry: 2026-09-15
   • Symbol      : NSE:NIFTY2691522500PE
   • ATM Spot    : 23200
   • Strike      : 22500 PE
   • ATM Offset  : -14 strikes (-700 pts from ATM)

📌 Date: 2026-09-24 | Expiry: 2026-09-15
   • Symbol      : NSE:NIFTY2691522500CE
   • ATM Spot    : 23200
   • Strike      : 22500 CE
   • ATM Offset  : -14 strikes (-700 pts from ATM)

📌 Date: 2026-09-24 | Expiry: 2026-09-15
   • Symbol      : NSE:NIFTY2691522550CE
   • ATM Spot    : 23200
   • Strike      : 22550 CE
   • ATM Offset  : -13 strikes (-650 pts from ATM)

📌 Date: 2026-09-24 | Expiry: 2026-09-15
   • Symbol      : NSE:NIFTY2691522550PE
   • ATM Spot    : 23200
   • Strike      : 22550 PE
   • ATM Offset  : -13 strikes (-650 pts from ATM)

📌 Date: 2026-09-24 | Expiry: 2026-09-15
   • Symbol      : NSE:NIFTY2691522600CE
   • ATM Spot    : 23200
   • Strike      : 22600 CE
   • ATM Offset  : 

oct 2028 spot download

In [3]:
import os
import time
import requests
import polars as pl
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration & Paths
TARGET_YEAR = 2018
TARGET_MONTH = 10

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
SPOT_DIR = os.path.join(BASE_DIR, "spot", f"year={TARGET_YEAR:04d}", f"month={TARGET_MONTH:02d}")
os.makedirs(SPOT_DIR, exist_ok=True)

SPOT_FILEPATH = os.path.join(SPOT_DIR, "NIFTY50.parquet")

# STANDARD HISTORY ENDPOINT (For Indices/Spot Data)
STANDARD_HISTORICAL_URL = "https://api-t1.fyers.in/data/history"
HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}

# October 2018 range
RANGE_FROM = "2018-10-01"
RANGE_TO = "2018-10-31"
SPOT_SYMBOL = "NSE:NIFTY50-INDEX"

print("===========================================================================")
print(f"🛡️ NIFTY 50 Spot Downloader Starting for: {TARGET_YEAR}-{TARGET_MONTH:02d}")
print("===========================================================================")

# Idempotency Check
if os.path.exists(SPOT_FILEPATH) and os.path.getsize(SPOT_FILEPATH) > 0:
    print(f"⚡ File already exists at: {SPOT_FILEPATH} (Skipped download)")
else:
    params = {
        "symbol": SPOT_SYMBOL,
        "resolution": "1",
        "date_format": "1",
        "range_from": RANGE_FROM,
        "range_to": RANGE_TO
    }

    max_retries = 8
    download_success = False

    for attempt in range(max_retries):
        try:
            res = requests.get(STANDARD_HISTORICAL_URL, headers=HEADERS, params=params, timeout=15)
            if res.status_code == 429:
                time.sleep(2.0 * (attempt + 1))
                continue

            res_data = res.json()
            candles = res_data.get("candles") or res_data.get("data", {}).get("candles", [])

            if res_data.get("s") == "ok" and candles:
                # Construct Polars DataFrame
                df_spot = pl.DataFrame(
                    candles,
                    schema=["epoch", "open", "high", "low", "close", "volume"],
                    orient="row"
                )

                # Schema transformation & integer scaling
                df_spot_transformed = df_spot.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    (pl.col("open") * 100).round(0).cast(pl.Int32).alias("open_i32"),
                    (pl.col("high") * 100).round(0).cast(pl.Int32).alias("high_i32"),
                    (pl.col("low") * 100).round(0).cast(pl.Int32).alias("low_i32"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("close_i32")
                ]).select(["timestamp", "open_i32", "high_i32", "low_i32", "close_i32"])

                # Save compressed Parquet
                df_spot_transformed.write_parquet(SPOT_FILEPATH, compression="zstd", compression_level=3)
                print(f"✅ Spot data successfully saved to: {SPOT_FILEPATH}")
                print(f"📊 Total Rows: {len(df_spot_transformed)} | File Size: {os.path.getsize(SPOT_FILEPATH) / 1024:.2f} KB")
                download_success = True
                break
            else:
                print(f"⚠️ Unexpected response: {res_data}")
                time.sleep(1.5)
        except Exception as e:
            print(f"❌ Error during download attempt {attempt+1}: {e}")
            time.sleep(1.5)

    if not download_success:
        print(f"❌ Failed to download spot data for {TARGET_YEAR}-{TARGET_MONTH:02d}")

Mounted at /content/drive
🛡️ NIFTY 50 Spot Downloader Starting for: 2018-10
✅ Spot data successfully saved to: /content/drive/MyDrive/fyers_backtest_data/spot/year=2018/month=10/NIFTY50.parquet
📊 Total Rows: 7875 | File Size: 123.08 KB


oct 2018 option expiry downloader

In [5]:
import os
import re
import time
import calendar
import requests
import polars as pl
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ----------------------------------------------------------------------
# 2. CONFIGURATION & CONSTANTS
# ----------------------------------------------------------------------
TARGET_YEAR = 2018
TARGET_MONTHS = [10]

BASE_DIR = "/content/drive/MyDrive/fyers_backtest_data"
OPTIONS_BASE_DIR = os.path.join(BASE_DIR, "options")

EXPIRY_DATES_URL = "https://api-t1.fyers.in/data/history/fno/expired/expiry-dates"
UNDERLYING_SYMBOLS_URL = "https://api-t1.fyers.in/data/history/fno/expired/underlying-symbols"
EXPIRED_CANDLES_URL = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

HEADERS = {"Authorization": f"{APP_ID}:{access_token}"}
MAX_WORKERS = 2

# ----------------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ----------------------------------------------------------------------
def fetch_json_with_retry(url, params, retries=3, delay=1.0):
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=3.0)
            if res.status_code == 200:
                return res.json()
            elif res.status_code in [429, 503]:
                time.sleep(delay * (attempt + 1))
        except Exception:
            time.sleep(0.5)
    return {}

def process_single_contract_day(symbol, date_str, target_folder, max_retries=3):
    opt_filename = f"{date_str}_{symbol.replace(':', '_')}.parquet"
    opt_filepath = os.path.join(target_folder, opt_filename)

    if os.path.exists(opt_filepath) and os.path.getsize(opt_filepath) > 0:
        return "skipped"

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": date_str,
        "range_to": date_str
    }

    for attempt in range(max_retries):
        try:
            # Strict 3-second timeout prevents network hanging
            res_obj = requests.get(EXPIRED_CANDLES_URL, headers=HEADERS, params=params, timeout=3.0)

            if res_obj.status_code == 429:
                time.sleep(1.0)
                continue

            res = res_obj.json()
            candles = res.get("candles") or res.get("data", {}).get("candles", [])

            if res.get("s") == "ok" and candles:
                match = re.search(r"(\d{5})(CE|PE)$", symbol)
                if not match:
                    return "error"

                strike_val = int(match.group(1))
                opt_type = match.group(2)

                df_opt = pl.DataFrame(
                    candles, schema=["epoch", "open", "high", "low", "close", "volume"], orient="row"
                )

                df_opt_reduced = df_opt.with_columns([
                    pl.from_epoch("epoch", time_unit="s")
                      .dt.replace_time_zone("UTC")
                      .dt.convert_time_zone("Asia/Kolkata")
                      .alias("timestamp"),
                    pl.lit(strike_val).cast(pl.Int32).alias("strike"),
                    pl.lit(opt_type).cast(pl.Categorical).alias("type"),
                    (pl.col("close") * 100).round(0).cast(pl.Int32).alias("price_i32")
                ]).select(["timestamp", "strike", "type", "price_i32"])

                df_opt_reduced.write_parquet(opt_filepath, compression="zstd", compression_level=3)
                return "saved"
            elif res.get("s") == "no_data" or not candles:
                return "no_data"
        except Exception:
            pass

    return "error"

def get_previous_monthly_expiry(yr, mo):
    prev_mo = 12 if mo == 1 else mo - 1
    prev_yr = yr - 1 if mo == 1 else yr
    last_day_prev = calendar.monthrange(prev_yr, prev_mo)[1]

    res = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{prev_yr:04d}-{prev_mo:02d}-01",
        "range_to": f"{prev_yr:04d}-{prev_mo:02d}-{last_day_prev:02d}",
        "date_format": "1"
    })
    if res.get("s") == "ok" and "data" in res and "expiry_dates" in res["data"]:
        exp_list = sorted(res["data"]["expiry_dates"].get("options", []))
        if exp_list:
            return exp_list[-1]
    return (date(yr, mo, 1) - timedelta(days=1)).strftime("%Y-%m-%d")

# ----------------------------------------------------------------------
# 4. DYNAMIC PIPELINE EXECUTION
# ----------------------------------------------------------------------
for target_month in TARGET_MONTHS:
    year_str = f"{TARGET_YEAR:04d}"
    month_str = f"{target_month:02d}"
    month_dir = os.path.join(OPTIONS_BASE_DIR, f"year={year_str}", f"month={month_str}")
    last_day_month = calendar.monthrange(TARGET_YEAR, target_month)[1]

    print(f"\n===========================================================================")
    print(f"🛡️ High-Accuracy Options Downloader Starting for: {year_str}-{month_str}")
    print(f"===========================================================================")

    spot_curr_path = os.path.join(BASE_DIR, "spot", f"year={year_str}", f"month={month_str}", "NIFTY50.parquet")
    if not os.path.exists(spot_curr_path):
        print(f"⚠️ Spot data missing at {spot_curr_path}. Ensure spot downloader ran!")
        continue

    df_spot = pl.read_parquet(spot_curr_path).with_columns(pl.col("timestamp").dt.date().alias("date"))

    # Dynamic Expiry Fetch
    res_exp = fetch_json_with_retry(EXPIRY_DATES_URL, {
        "symbol": "NSE:NIFTY50-INDEX",
        "range_from": f"{year_str}-{month_str}-01",
        "range_to": f"{year_str}-{month_str}-{last_day_month:02d}",
        "date_format": "1"
    })

    if res_exp.get("s") == "ok" and res_exp.get("data", {}).get("expiry_dates", {}).get("options"):
        api_expiries = sorted(res_exp["data"]["expiry_dates"]["options"])
        print(f"✅ Dynamic Expiries Returned by FYERS for {year_str}-{month_str}: {api_expiries}")

        monthly_expiry_date = api_expiries[-1]
        weekly_expiries = api_expiries[:-1]

        folder_map = {}
        for idx, exp_date in enumerate(weekly_expiries):
            folder_map[exp_date] = os.path.join(month_dir, f"week{idx + 1}")
        folder_map[monthly_expiry_date] = os.path.join(month_dir, "monthly")

        for folder_path in folder_map.values():
            os.makedirs(folder_path, exist_ok=True)

        prev_expiry = get_previous_monthly_expiry(TARGET_YEAR, target_month)

        for exp_date in api_expiries:
            target_folder = folder_map[exp_date]
            folder_label = os.path.basename(target_folder)

            sym_res = fetch_json_with_retry(UNDERLYING_SYMBOLS_URL, {
                "symbol": "NSE:NIFTY50-INDEX",
                "expiry_date": exp_date
            })

            raw_options = sym_res.get("data", {}).get("contracts", {}).get("options", []) if sym_res.get("s") == "ok" else []

            start_dt = pl.lit(prev_expiry).str.to_date()
            end_dt = pl.lit(exp_date).str.to_date()

            trading_dates = df_spot.filter(
                (pl.col("date") > start_dt) & (pl.col("date") <= end_dt)
            )["date"].unique().sort()

            if len(trading_dates) == 0:
                prev_expiry = exp_date
                continue

            download_tasks = []
            for current_date in trading_dates:
                date_str = current_date.strftime("%Y-%m-%d")
                day_spot = df_spot.filter(pl.col("date") == current_date)
                if day_spot.is_empty(): continue

                day_open_spot = day_spot["open_i32"][0] / 100.0 if "open_i32" in day_spot.columns else day_spot["close_i32"].mean() / 100.0
                day_atm = int(round(day_open_spot / 50.0) * 50)
                day_strikes = set(range(day_atm - (15 * 50), day_atm + (16 * 50), 50))

                for symbol_name in raw_options:
                    match = re.search(r"(\d{5})(CE|PE)$", symbol_name)
                    if match and int(match.group(1)) in day_strikes:
                        download_tasks.append((symbol_name, date_str, target_folder))

            saved_cnt, skipped_cnt, error_cnt, no_data_cnt = 0, 0, 0, 0
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(process_single_contract_day, sym, d_str, folder)
                    for sym, d_str, folder in download_tasks
                ]
                for future in as_completed(futures):
                    st = future.result()
                    if st == "saved": saved_cnt += 1
                    elif st == "skipped": skipped_cnt += 1
                    elif st == "no_data": no_data_cnt += 1
                    else: error_cnt += 1

            print(f"⚡ {folder_label.upper()} ({exp_date}) Finished ➔ Saved: {saved_cnt} | Skipped: {skipped_cnt} | NoData: {no_data_cnt} | Errors: {error_cnt}", flush=True)
            prev_expiry = exp_date
    else:
        print(f"❌ Expiry API response empty or unavailable for {year_str}-{month_str}. (FYERS F&O historical data depth limit reached)")

Mounted at /content/drive

🛡️ High-Accuracy Options Downloader Starting for: 2018-10
✅ Dynamic Expiries Returned by FYERS for 2018-10: ['2018-10-25']


KeyboardInterrupt: 

In [11]:
import requests
import polars as pl

# -------------------------------------------------------------------
# TEST: 2019 Expired Options 1-Minute Candle Data Retrieval
# -------------------------------------------------------------------

# Uses the APP_ID and access_token generated from your auth script
headers = {
    "Authorization": f"{APP_ID}:{access_token}"
}

expired_fno_url = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

# Target Symbol: Feb 28, 2019 Monthly Expiry (10800 CE)
params_2019 = {
    "symbol": "NSE:NIFTY19FEB10800CE",
    "resolution": "1",                # 1-minute resolution
    "date_format": "1",               # YYYY-MM-DD
    "range_from": "2019-02-28",
    "range_to": "2019-02-28"
}

print("⚡ Testing 1-minute candles for Feb 2019 Expired Option...")

res_2019 = requests.get(expired_fno_url, headers=headers, params=params_2019).json()

if res_2019.get("s") == "ok":
    candles = res_2019.get("candles", [])
    if candles:
        df_2019 = pl.DataFrame(
            candles,
            schema=["timestamp", "open", "high", "low", "close", "volume"],
            orient="row"
        )
        df_2019 = df_2019.with_columns(pl.from_epoch("timestamp", time_unit="s"))
        print("\n✅ SUCCESS! 2019 1-minute candles exist:")
        print(df_2019.head(10))
    else:
        print("\n⚠️ API returned 'ok', but candle list is EMPTY. (2019 data is not archived on FYERS backend).")
else:
    print(f"\n❌ Error / Failed Response from FYERS API:\n{res_2019}")

⚡ Testing 1-minute candles for Feb 2019 Expired Option...

✅ SUCCESS! 2019 1-minute candles exist:
shape: (10, 6)
┌─────────────────────┬───────┬───────┬───────┬───────┬────────┐
│ timestamp           ┆ open  ┆ high  ┆ low   ┆ close ┆ volume │
│ ---                 ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---    │
│ datetime[μs]        ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ i64    │
╞═════════════════════╪═══════╪═══════╪═══════╪═══════╪════════╡
│ 2019-02-28 03:45:00 ┆ 73.95 ┆ 74.0  ┆ 56.25 ┆ 56.7  ┆ 208278 │
│ 2019-02-28 03:46:00 ┆ 55.55 ┆ 60.65 ┆ 55.15 ┆ 60.1  ┆ 326550 │
│ 2019-02-28 03:47:00 ┆ 60.2  ┆ 62.5  ┆ 60.2  ┆ 61.35 ┆ 186600 │
│ 2019-02-28 03:48:00 ┆ 61.4  ┆ 65.6  ┆ 61.4  ┆ 65.6  ┆ 136800 │
│ 2019-02-28 03:49:00 ┆ 65.25 ┆ 65.65 ┆ 61.2  ┆ 61.6  ┆ 180675 │
│ 2019-02-28 03:50:00 ┆ 60.8  ┆ 61.6  ┆ 57.5  ┆ 57.5  ┆ 155250 │
│ 2019-02-28 03:51:00 ┆ 57.9  ┆ 58.6  ┆ 53.7  ┆ 55.85 ┆ 173925 │
│ 2019-02-28 03:52:00 ┆ 56.05 ┆ 56.2  ┆ 52.7  ┆ 55.55 ┆ 182025 │
│ 2019-02-28 03:53:00 ┆ 55.7  ┆ 60.35 ┆ 5

In [12]:
import calendar
from datetime import date, timedelta
import requests
import polars as pl

# -------------------------------------------------------------------
# 1. EXPIRY CALCULATOR FUNCTION FOR NIFTY
# -------------------------------------------------------------------

def get_nifty_expiries_for_month(year: int, month: int):
    """
    Calculates all Nifty options expiry dates for a specified year and month.
    Handles Thursday expiries (and accounts for standard monthly/weekly patterns).
    """
    expiries = []
    num_days = calendar.monthrange(year, month)[1]

    # Iterate through all days in the month to find Thursday expiries
    for day in range(1, num_days + 1):
        d = date(year, month, day)
        # 3 represents Thursday (0=Monday, 1=Tuesday, ..., 3=Thursday)
        if d.weekday() == 3:
            expiries.append(d)

    return expiries

def get_fyers_option_symbol(expiry_date: date, strike: int, opt_type: str = "CE") -> str:
    """
    Constructs FYERS Option Symbol string.
    - Monthly Expiry: NSE:NIFTY{YY}{MMM}{STRIKE}{TYPE} (e.g., NSE:NIFTY19FEB10800CE)
    - Weekly Expiry:  NSE:NIFTY{YY}{M_CHAR}{DD}{STRIKE}{TYPE} (e.g., NSE:NIFTY24O1725000CE)
    """
    yy = expiry_date.strftime("%y")

    # Check if this expiry date is the last Thursday (Monthly Expiry)
    num_days = calendar.monthrange(expiry_date.year, expiry_date.month)[1]
    last_thursday = max(
        date(expiry_date.year, expiry_date.month, d)
        for d in range(1, num_days + 1)
        if date(expiry_date.year, expiry_date.month, d).weekday() == 3
    )

    is_monthly = (expiry_date == last_thursday)

    if is_monthly:
        mmm = expiry_date.strftime("%b").upper()
        symbol = f"NSE:NIFTY{yy}{mmm}{strike}{opt_type}"
    else:
        # FYERS Weekly month codes: 1-9 for Jan-Sep, O for Oct, N for Nov, D for Dec
        month_codes = {1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'O', 11:'N', 12:'D'}
        m_char = month_codes[expiry_date.month]
        dd = expiry_date.strftime("%d")
        symbol = f"NSE:NIFTY{yy}{m_char}{dd}{strike}{opt_type}"

    return symbol

# -------------------------------------------------------------------
# 2. RUN EXPIRY FINDER & API VERIFICATION
# -------------------------------------------------------------------

# Target Year & Month to inspect
TARGET_YEAR = 2019
TARGET_MONTH = 2
TEST_STRIKE = 10800

expiries = get_nifty_expiries_for_month(TARGET_YEAR, TARGET_MONTH)

print(f"📅 Expiry Dates found for {calendar.month_name[TARGET_MONTH]} {TARGET_YEAR}:")
for exp in expiries:
    sym = get_fyers_option_symbol(exp, TEST_STRIKE, "CE")
    print(f" - Date: {exp} | Generated FYERS Symbol: {sym}")

# -------------------------------------------------------------------
# 3. VERIFY CANDLE DATA VIA FYERS API FOR EACH EXPIRY IN THE MONTH
# -------------------------------------------------------------------

headers = {
    "Authorization": f"{APP_ID}:{access_token}"
}
expired_fno_url = "https://api-t1.fyers.in/data/history/fno/expired/historical-data"

print("\n⚡ Verifying 1-minute candle availability across month expiries...")

for exp in expiries:
    expiry_str = exp.strftime("%Y-%m-%d")
    symbol = get_fyers_option_symbol(exp, TEST_STRIKE, "CE")

    params = {
        "symbol": symbol,
        "resolution": "1",
        "date_format": "1",
        "range_from": expiry_str,
        "range_to": expiry_str
    }

    res = requests.get(expired_fno_url, headers=headers, params=params).json()

    if res.get("s") == "ok":
        candles_count = len(res.get("candles", []))
        print(f"✅ {expiry_str} [{symbol}] -> {candles_count} candles retrieved.")
    else:
        print(f"❌ {expiry_str} [{symbol}] -> Failed/Error: {res.get('message', res)}")

📅 Expiry Dates found for February 2019:
 - Date: 2019-02-07 | Generated FYERS Symbol: NSE:NIFTY1920710800CE
 - Date: 2019-02-14 | Generated FYERS Symbol: NSE:NIFTY1921410800CE
 - Date: 2019-02-21 | Generated FYERS Symbol: NSE:NIFTY1922110800CE
 - Date: 2019-02-28 | Generated FYERS Symbol: NSE:NIFTY19FEB10800CE

⚡ Verifying 1-minute candle availability across month expiries...
❌ 2019-02-07 [NSE:NIFTY1920710800CE] -> Failed/Error: Invalid input
✅ 2019-02-14 [NSE:NIFTY1921410800CE] -> 375 candles retrieved.
✅ 2019-02-21 [NSE:NIFTY1922110800CE] -> 375 candles retrieved.
✅ 2019-02-28 [NSE:NIFTY19FEB10800CE] -> 375 candles retrieved.
